In [1]:
import numpy as np

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the vertical depolarization factor (Delta_z) for a spheroid.
    Vectorized to handle scalar or numpy arrays of aspect ratios.
    """
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    # Spheres (m == 1)
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    # Prolate (Columns, m > 1)
    mask_prolate = (m > 1.0)
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    # Oblate (Plates, m < 1)
    mask_oblate = (m < 1.0)
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta


def compute_polarizability(frequency, aspect_ratio=1.0):
    """
    Computes the raw complex polarizabilities (alpha_h, alpha_v) for ice crystals.
    These are used directly in the M_raw matrix and emission calculations.
    """
    # Complex relative permittivity of ice 
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # Get the vertical depolarization factor
    delta_z = compute_depolarization_factor(aspect_ratio)
    
    # Calculate raw polarizabilities (matches CLASS paper Eq 2)
    # The 4*pi is kept here to cancel with the physical volume later
    alpha_v = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta_z))
    alpha_h = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta_z) / 2))
    
    return alpha_h, alpha_v

In [2]:
def compute_raw_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    """
    Computes the M11 (intensity) and M21 (Q polarization) components 
    of the raw phase matrix for horizontally aligned spheroids.
    """
    # Calculate the complex cross-term shared by both equations
    cross_term = alpha_v * np.sin(theta_grid) * np.sin(delta) - \
                 alpha_h * np.cos(theta_grid) * np.cos(delta) * np.cos(phi_grid)
                 
    # Take the absolute squares needed for the matrix
    abs_h2 = np.abs(alpha_h)**2
    abs_cross2 = np.abs(cross_term)**2
    abs_h_cos_phi2 = np.abs(alpha_h * np.cos(phi_grid))**2
    
    # Evaluate M11 and M21
    M11 = 0.5 * (abs_h2 + abs_cross2 + abs_h_cos_phi2)
    M21 = 0.5 * (abs_cross2 + abs_h_cos_phi2 - abs_h2)
    
    return M11, M21

In [3]:
import numpy as np

def compute_geometry(theta_grid, phi_grid, delta):
    """
    Computes the scattering angle (Theta) and rotation angle (psi)
    for every coordinate in the sky grid.
    
    Parameters:
    theta_grid : 2D array of incoming zenith angles (radians)
    phi_grid   : 2D array of incoming azimuth angles (radians)
    delta      : Float, telescope elevation angle (radians)
    
    Returns:
    Theta_grid : 2D array of scattering angles
    psi_grid   : 2D array of rotation angles
    """
    # 1. Scattering Angle (Theta)
    cos_Theta = np.sin(delta) * np.cos(theta_grid) + \
                np.cos(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    
    # Clip to strictly [-1, 1] to prevent floating-point NaN errors in arccos
    cos_Theta = np.clip(cos_Theta, -1.0, 1.0)
    Theta_grid = np.arccos(cos_Theta)
    
    # 2. Rotation Angle (psi)
    # Derived from spherical law of sines and cosines
    y = np.sin(theta_grid) * np.sin(phi_grid)
    x = (np.sin(delta) * cos_Theta - np.cos(theta_grid)) / np.cos(delta)
    
    psi_grid = np.arctan2(y, x)
    
    return Theta_grid, psi_grid

In [4]:
import numpy as np

# --- 1. Setup Physical Constants & Parameters ---
c = 3e8                 # Speed of light (m/s)
frequency = 90e9        # Observation frequency (90 GHz)
R_e = 6371e3            # Earth radius in meters
T_ground = 270.0        # Physical temperature of the ground (Kelvin)
T_sky_zenith = 10.0     # Example temperature of the cold sky at zenith (Kelvin)

# Telescope & Particle Parameters
delta = np.radians(45)  # Telescope elevation (45 degrees)
aspect_ratio = 1      # Aspect ratio of the ice crystal (e.g., 0.1 for a plate)
D = 100e-6              # Geometric mean diameter of ice crystals (100 micrometers)

# Cloud Profile (Example: a 2 km thick cloud from 6 km to 8 km)
# You will replace these with your actual ERA5 or simulated atmospheric profile
dz = 100.0                             # Layer thickness (100 meters)
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5                              # Constant number density (particles per m^3) for testing

# --- 2. Setup Integration Grid ---
N_theta, N_phi = 100, 100
theta = np.linspace(0, np.pi, N_theta)
phi = np.linspace(0, 2*np.pi, N_phi)
dtheta = theta[1] - theta[0]
dphi = phi[1] - phi[0]

# Create 2D meshgrids
theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')

# --- 3. Pre-compute Geometry and Matrices ---
# (Assuming these functions are imported from your module)
Theta_grid, psi_grid = compute_geometry(theta_grid, phi_grid, delta)

# Get raw polarizabilities
alpha_h, alpha_v = compute_polarizability(frequency, aspect_ratio)

# Get raw phase matrix components
M11_grid, M21_grid = compute_raw_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)

# --- 4. Layer-by-Layer Integration ---
Q_detected = 0.0

for z in altitude_layers:
    
    # A. Calculate the horizon dip angle for this specific altitude
    theta_h = np.sqrt(2 * z / R_e)
    
    # Define the absolute theta boundary (assuming theta=0 is straight up)
    theta_boundary = (np.pi / 2) + theta_h
    
    # B. Build the Temperature Grid (T_in)
    # Simple model: sky temperature increases slightly toward the horizon (secant law)
    # Clip the angle to prevent division by zero near the horizon
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    
    # Assign T_ground if looking below the boundary, T_sky_profile if looking above
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)
    
    # C. Calculate the Layer Prefactor
    # C_layer = n(z) * airmass * dz * (2pi * nu / c)^4 * (D/2)^6
    C_layer = n_z * (1 / np.sin(delta)) * dz * (2 * np.pi * frequency / c)**4 * (D / 2)**6
    
    # D. The Integral (Simplified, ignoring psi rotation as requested)
    # If you ever want to add the telescope rotation frame back in, 
    # just multiply this by np.cos(2 * psi_grid)
    integrand_Q = M21_grid * np.cos(2 * psi_grid) * T_in * np.sin(theta_grid)
    
    # Sum over the grids and multiply by differential steps
    dQ_layer = C_layer * np.sum(integrand_Q) * dtheta * dphi
    
    # Accumulate total polarization
    Q_detected += dQ_layer

print(f"Total Detected Q Polarization: {Q_detected:.4e} Kelvin")

Total Detected Q Polarization: 1.5205e-04 Kelvin


In [5]:
print(theta_boundary*180/np.pi)  # Print the boundary angle in degrees for verification

92.85329958577138


Test

In [6]:
# %%
import numpy as np

import numpy as np

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) for the SYMMETRY axis 
    of a spheroid based on its aspect ratio.
    """
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    mask_prolate = (m > 1.0)  # Columns
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    mask_oblate = (m < 1.0)   # Plates
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta


def compute_polarizability(frequency, aspect_ratio=1.0):
    """
    Computes the complex polarizabilities (alpha_h, alpha_v) for ice crystals,
    properly accounting for the aerodynamic falling orientation of plates vs columns.
    Maintains the 4*pi scaling required for the Phase Matrix computation.
    """
    # 1. Complex relative permittivity of ice
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # 2. Get the depolarization factor for the SYMMETRY axis
    delta = compute_depolarization_factor(aspect_ratio)
    
    # 3. Calculate the polarizabilities native to the particle's own shape
    A_par = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta))
    A_perp = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta) / 2))
    
    # 4. Map the particle axes to the Earth's axes based on aerodynamics
    if aspect_ratio == 1.0:
        # SPHERE: Completely isotropic
        alpha_v = A_par
        alpha_h = A_par
        
    elif aspect_ratio < 1.0:
        # PLATES (Oblate Spheroids)
        # Aerodynamics: Falls flat. The short symmetry axis points vertically.
        # The identical transverse axes form the horizontal plane.
        alpha_v = A_par
        alpha_h = A_perp
        
    else:
        # COLUMNS (Prolate Spheroids)
        # Aerodynamics: Falls flat and spins randomly in the azimuth (helicopter).
        # The vertical axis sees the narrow transverse cross-section.
        # The horizontal plane averages the long symmetry axis and the short transverse axis.
        alpha_v = A_perp
        alpha_h = (A_par + A_perp) / 2.0
        
    return alpha_h, alpha_v

# %%
def compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    """
    METHOD 1: Direct Earth-Frame Projection.
    Computes the M11 (intensity) and M21 (Q polarization) components 
    natively in the telescope's referential frame. 
    """
    abs_h2 = np.abs(alpha_h)**2
    abs_v2 = np.abs(alpha_v)**2
    
    # 1. True Horizontal Intensity (I_H)
    I_H = abs_h2 * (1.0 - (np.sin(theta_grid) * np.sin(phi_grid))**2)
    
    # 2. True Vertical Intensity (I_V)
    # The projection of the incoming ray onto the vertical sensor
    ray_proj = alpha_v * np.cos(delta) * np.cos(theta_grid) - \
               alpha_h * np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
               
    I_V = (abs_h2 * np.sin(delta)**2 + abs_v2 * np.cos(delta)**2) - np.abs(ray_proj)**2
    
    # 3. Final Phase Matrix
    M11_earth = 0.5 * (I_V + I_H)
    M21_earth = 0.5 * (I_V - I_H)
    
    return M11_earth, M21_earth

# %%
def compute_geometry(theta_grid, phi_grid, delta):
    """
    Computes the scattering angle (Theta) and rotation angle (psi)
    for every coordinate in the sky grid.
    """
    # 1. Scattering Angle (Theta)
    cos_Theta = np.sin(delta) * np.cos(theta_grid) + \
                np.cos(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    
    cos_Theta = np.clip(cos_Theta, -1.0, 1.0)
    Theta_grid = np.arccos(cos_Theta)
    
    # 2. Rotation Angle (psi)
    y = np.sin(theta_grid) * np.sin(phi_grid)
    x = np.cos(theta_grid) * np.cos(delta) - np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    
    psi_grid = np.arctan2(y, x)
    
    return Theta_grid, psi_grid

# %%
# --- Setup Physical Constants & Parameters ---
c = 3e8                 
frequency = 90e9        
R_e = 6371e3            
T_ground = 270.0        
T_sky_zenith = 10.0     

delta = np.radians(45)  
aspect_ratio = 4      # columns
D = 100e-6              
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5                              

# --- Setup Integration Grid ---
N_theta, N_phi = 100, 100
theta = np.linspace(0, np.pi, N_theta)
phi = np.linspace(0, 2*np.pi, N_phi)
dtheta = theta[1] - theta[0]
dphi = phi[1] - phi[0]
theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')

# --- Pre-compute Geometry and Matrices ---
Theta_grid, psi_grid = compute_geometry(theta_grid, phi_grid, delta)
alpha_h, alpha_v = compute_polarizability(frequency, aspect_ratio)

# Get the phase matrix natively in the Earth frame!
M11_earth, M21_earth = compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)

# --- Layer-by-Layer Integration ---
Q_detected_method_1 = 0.0

for z in altitude_layers:
    
    theta_h = np.sqrt(2 * z / R_e)
    theta_boundary = (np.pi / 2) + theta_h
    
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)
    
    C_layer = n_z * (1 / np.sin(delta)) * dz * (2 * np.pi * frequency / c)**4 * (D / 2)**6
    
    # =========================================================================
    # METHOD 1: The Direct Earth-Frame Method (Highly Recommended)
    # Because M21_earth is already projected onto the telescope sensors,
    # there is NO psi rotation matrix required here. 
    # =========================================================================
    integrand_Q_m1 = M21_earth * T_in * np.sin(theta_grid)
    dQ_layer_m1 = C_layer * np.sum(integrand_Q_m1) * dtheta * dphi
    Q_detected_method_1 += dQ_layer_m1
    
    # =========================================================================
    # METHOD 2: The Textbook Framework (Conceptual equivalent)
    # If we had calculated the matrix 'F_21' purely inside the 2D scattering 
    # plane, the math demands we multiply by the rotation matrix cos(2*psi).
    # 
    # (Note: To make this run, F_21_scattering_plane would be mathematically 
    # reverse-engineered from M21_earth, but this shows exactly where psi 
    # goes in the textbook architecture).
    # =========================================================================
    # F_21_scattering_plane = ... (Complex tensor rotation of alpha_h and alpha_v)
    # integrand_Q_m2 = F_21_scattering_plane * np.cos(2 * psi_grid) * T_in * np.sin(theta_grid)
    # dQ_layer_m2 = C_layer * np.sum(integrand_Q_m2) * dtheta * dphi

print(f"Total Detected Q Polarization (Method 1): {Q_detected_method_1:.4e} Kelvin")
print(f"Horizon angle for highest layer: {theta_boundary*180/np.pi:.2f} degrees")


Total Detected Q Polarization (Method 1): -1.0816e-04 Kelvin
Horizon angle for highest layer: 92.85 degrees


In [7]:
# %%
import numpy as np

def compute_depolarization_factor(aspect_ratio):
    """Calculates the vertical depolarization factor (Delta_z) for a spheroid."""
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    mask_prolate = (m > 1.0)
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    mask_oblate = (m < 1.0)
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta

def compute_polarizability(frequency, aspect_ratio=1.0):
    """Computes the raw complex polarizabilities (alpha_h, alpha_v)."""
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    delta_z = compute_depolarization_factor(aspect_ratio)
    
    alpha_v = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta_z))
    alpha_h = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta_z) / 2))
    return alpha_h, alpha_v

def compute_geometry(theta_grid, phi_grid, delta):
    """Computes the Scattering Angle (Theta) and Rotation Angle (psi)."""
    cos_Theta = np.sin(delta) * np.cos(theta_grid) + \
                np.cos(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    Theta_grid = np.arccos(np.clip(cos_Theta, -1.0, 1.0))
    
    y = np.sin(theta_grid) * np.sin(phi_grid)
    x = np.cos(theta_grid) * np.cos(delta) - np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    psi_grid = np.arctan2(y, x)
    
    return Theta_grid, psi_grid

# ====================================================================
# METHOD 1: DIRECT EARTH FRAME (Fast & Elegant)
# ====================================================================
def compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    
    abs_h2 = np.abs(alpha_h)**2
    abs_v2 = np.abs(alpha_v)**2
    
    # Direct projection onto Telescope Horizontal and Vertical sensors
    I_H = abs_h2 * (1.0 - (np.sin(theta_grid) * np.sin(phi_grid))**2)
    
    ray_proj = alpha_v * np.cos(delta) * np.cos(theta_grid) - \
               alpha_h * np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    I_V = (abs_h2 * np.sin(delta)**2 + abs_v2 * np.cos(delta)**2) - np.abs(ray_proj)**2
    
    M21_earth = 0.5 * (I_V - I_H)
    return M21_earth

# ====================================================================
# METHOD 2: TEXTBOOK SCATTERING PLANE (Complex 3D Vector Math)
# ====================================================================
def compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta, psi_grid):
    
    # 1. Define Propagation Vectors (Shape: N_theta, N_phi, 3)
    # Incident ray travels FROM the sky TO the origin
    s_i = np.stack([-np.sin(theta_grid) * np.cos(phi_grid), 
                    -np.sin(theta_grid) * np.sin(phi_grid), 
                    -np.cos(theta_grid)], axis=-1)
    
    # Scattered ray travels FROM the origin TO the telescope
    s_s = np.array([np.cos(delta), 0.0, np.sin(delta)])
    s_s = np.broadcast_to(s_s, s_i.shape)
    
    # 2. Build the 2D Scattering Plane Unit Vectors
    # Perpendicular vector (Normal to scattering plane)
    n_scat = np.cross(s_i, s_s, axis=-1)
    norm = np.linalg.norm(n_scat, axis=-1, keepdims=True)
    norm = np.where(norm < 1e-12, 1e-12, norm) # Prevent division by zero
    e_perp = n_scat / norm
    
    # Parallel vectors (Inside the scattering plane)
    e_par_i = np.cross(s_i, e_perp, axis=-1)
    e_par_s = np.cross(s_s, e_perp, axis=-1)
    
    # 3. Apply the Earth-locked Ice Crystal Tensor
    def apply_tensor(v):
        return np.stack([alpha_h * v[..., 0], alpha_h * v[..., 1], alpha_v * v[..., 2]], axis=-1)
    
    # 4. Calculate Bohren & Huffman Amplitude Matrix Elements (S1, S2, S3, S4)
    S1 = np.sum(e_perp * apply_tensor(e_perp), axis=-1)
    S2 = np.sum(e_par_s * apply_tensor(e_par_i), axis=-1)
    S3 = np.sum(e_par_s * apply_tensor(e_perp), axis=-1)
    S4 = np.sum(e_perp * apply_tensor(e_par_i), axis=-1)
    
    # 5. Calculate Stokes Phase Matrix inside the local scattering plane (F_matrix)
    # Note: Because the particle is aligned, cross-polarization (F31) is NOT zero!
    F21 = 0.5 * (np.abs(S2)**2 + np.abs(S3)**2 - np.abs(S1)**2 - np.abs(S4)**2)
    F31 = np.real(S2 * np.conj(S4) + S3 * np.conj(S1))
    
    # 6. Apply Rotation Matrix R(psi) to twist the light back to the Telescope
    M21_rotated = F21 * np.cos(2 * psi_grid) - F31 * np.sin(2 * psi_grid)
    
    return M21_rotated

# %%
# --- Execute Simulation ---
c = 3e8                 
frequency = 90e9        
R_e = 6371e3            
T_ground = 270.0        
T_sky_zenith = 10.0     

delta = np.radians(45)  
aspect_ratio = 4      # columns
D = 100e-6              
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5                              

N_theta, N_phi = 100, 100
theta = np.linspace(0, np.pi, N_theta)
phi = np.linspace(0, 2*np.pi, N_phi)
dtheta, dphi = theta[1] - theta[0], phi[1] - phi[0]
theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')

# Pre-compute Physics
Theta_grid, psi_grid = compute_geometry(theta_grid, phi_grid, delta)
alpha_h, alpha_v = compute_polarizability(frequency, aspect_ratio)

# Generate the Matrix grids using BOTH methods
M21_method_1 = compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)
M21_method_2 = compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta, psi_grid)

# Integrate
Q_detected_M1 = 0.0
Q_detected_M2 = 0.0

for z in altitude_layers:
    theta_h = np.sqrt(2 * z / R_e)
    theta_boundary = (np.pi / 2) + theta_h
    
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)
    
    C_layer = n_z * (1 / np.sin(delta)) * dz * (2 * np.pi * frequency / c)**4 * (D / 2)**6
    
    # Method 1 Integral (No Psi Rotation needed)
    Q_detected_M1 += C_layer * np.sum(M21_method_1 * T_in * np.sin(theta_grid)) * dtheta * dphi
    
    # Method 2 Integral (Psi rotation already applied inside the function)
    Q_detected_M2 += C_layer * np.sum(M21_method_2 * T_in * np.sin(theta_grid)) * dtheta * dphi

print("=== RESULTS ===")
print(f"Q Polarization (Method 1 - Earth Frame):   {Q_detected_M1:.6e} K")
print(f"Q Polarization (Method 2 - Textbook):      {Q_detected_M2:.6e} K")
print(f"Absolute Difference:                       {abs(Q_detected_M1 - Q_detected_M2):.6e} K")

=== RESULTS ===
Q Polarization (Method 1 - Earth Frame):   3.517107e-04 K
Q Polarization (Method 2 - Textbook):      3.517107e-04 K
Absolute Difference:                       5.421011e-20 K


In [33]:
import numpy as np

import numpy as np

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) for the SYMMETRY axis 
    of a spheroid based on its aspect ratio.
    """
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    mask_prolate = (m > 1.0)  # Columns
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    mask_oblate = (m < 1.0)   # Plates
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta


import numpy as np

def compute_polarizability(frequency, aspect_ratio=1.0):
    """
    Computes the complex polarizabilities (alpha_h, alpha_v) for ice crystals.
    Constructs an effective complex alpha_h for columns to mathematically respect 
    the incoherent azimuthal averaging of the absolute squares.
    """
    # 1. Complex relative permittivity of ice
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # 2. Get intrinsic properties
    delta = compute_depolarization_factor(aspect_ratio)
    A_par = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta))
    A_perp = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta) / 2.0))
    
    # 3. Map to Earth-frame
    if aspect_ratio == 1.0:
        # SPHERE
        alpha_v = A_par
        alpha_h = A_par
        
    elif aspect_ratio < 1.0:
        # PLATES (Horizontal plane is pure transverse)
        alpha_v = A_par
        alpha_h = A_perp
        
    else:
        # COLUMNS (Helicopter Spin)
        alpha_v = A_perp
        
        # Target 1: Averaged Intensity (Scattering)
        target_abs2_h = (np.abs(A_par)**2 + np.abs(A_perp)**2) / 2.0
        
        # Target 2: Averaged Absorption (Emission)
        target_imag_h = (np.imag(A_par) + np.imag(A_perp)) / 2.0
        
        # Construct the effective complex alpha_h
        real_h = np.sqrt(np.maximum(0, target_abs2_h - target_imag_h**2))
        alpha_h = real_h + 1j * target_imag_h
        
    return alpha_h, alpha_v

# ====================================================================
# METHOD 1: DIRECT EARTH FRAME (Fast & Elegant)
# ====================================================================
def compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    abs_h2 = np.abs(alpha_h)**2
    abs_v2 = np.abs(alpha_v)**2
    
    I_H = abs_h2 * (1.0 - (np.sin(theta_grid) * np.sin(phi_grid))**2)
    
    ray_proj = alpha_v * np.cos(delta) * np.cos(theta_grid) - \
               alpha_h * np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    I_V = (abs_h2 * np.sin(delta)**2 + abs_v2 * np.cos(delta)**2) - np.abs(ray_proj)**2
    
    M21_earth = 0.5 * (I_V - I_H)
    return M21_earth

# ====================================================================
# METHOD 2: TEXTBOOK SCATTERING PLANE (Corrected Vector Math)
# ====================================================================
def compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    
    # 1. Define Propagation Vectors (k_i and k_s)
    s_i = np.stack([-np.sin(theta_grid) * np.cos(phi_grid), 
                    -np.sin(theta_grid) * np.sin(phi_grid), 
                    -np.cos(theta_grid)], axis=-1)
    
    s_s = np.array([np.cos(delta), 0.0, np.sin(delta)])
    s_s = np.broadcast_to(s_s, s_i.shape)
    
    # 2. Build the 2D Scattering Plane Unit Vectors
    n_scat = np.cross(s_i, s_s, axis=-1)
    norm = np.linalg.norm(n_scat, axis=-1, keepdims=True)
    norm = np.where(norm < 1e-12, 1e-12, norm) 
    e_perp = n_scat / norm
    
    # CORRECTED: Strict Right-Hand Rule (e_par = e_perp x k)
    e_par_i = np.cross(e_perp, s_i, axis=-1)
    e_par_s = np.cross(e_perp, s_s, axis=-1)
    
    # 3. Apply the Earth-locked Ice Crystal Tensor
    def apply_tensor(v):
        return np.stack([alpha_h * v[..., 0], alpha_h * v[..., 1], alpha_v * v[..., 2]], axis=-1)
    
    # 4. Calculate Bohren & Huffman Amplitude Matrix Elements
    S1 = np.sum(e_perp * apply_tensor(e_perp), axis=-1)
    S2 = np.sum(e_par_s * apply_tensor(e_par_i), axis=-1)
    S3 = np.sum(e_par_s * apply_tensor(e_perp), axis=-1)
    S4 = np.sum(e_perp * apply_tensor(e_par_i), axis=-1)
    
    # 5. Stokes Phase Matrix inside the local scattering plane (F)
    F21 = 0.5 * (np.abs(S2)**2 + np.abs(S3)**2 - np.abs(S1)**2 - np.abs(S4)**2)
    F31 = np.real(S2 * np.conj(S4) + S3 * np.conj(S1))
    
    # 6. EXACT Rotation: Bypass spherical trig and use exact 3D Dot Products
    e_v = np.array([-np.sin(delta), 0.0, np.cos(delta)])
    e_v = np.broadcast_to(e_v, s_i.shape)
    
    # Project the telescope vertical sensor onto the scattering plane basis
    cos_psi = np.sum(e_v * e_par_s, axis=-1)
    sin_psi = -np.sum(e_v * e_perp, axis=-1)
    
    # Apply double-angle formulas for the Mueller rotation matrix R(psi)
    cos_2psi = cos_psi**2 - sin_psi**2
    sin_2psi = 2 * sin_psi * cos_psi
    
    M21_rotated = F21 * cos_2psi - F31 * sin_2psi
    
    return M21_rotated

# ====================================================================
# EXECUTE SIMULATION
# ====================================================================
c = 3e8                 
frequency = 90e9        
R_e = 6371e3            
T_ground = 270.0        
T_sky_zenith = 10.0     

delta = np.radians(45)  
aspect_ratio = 1      # Plates
D = 100e-6              
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5                              

N_theta, N_phi = 100, 100
theta = np.linspace(0, np.pi, N_theta)
phi = np.linspace(0, 2*np.pi, N_phi)
dtheta, dphi = theta[1] - theta[0], phi[1] - phi[0]
theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')

# Pre-compute Physics
alpha_h, alpha_v = compute_polarizability(frequency, aspect_ratio)

# Generate the Matrix grids using BOTH methods
M21_method_1 = compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)
M21_method_2 = compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)

# Integrate
Q_detected_M1 = 0.0
Q_detected_M2 = 0.0

for z in altitude_layers:
    theta_h = np.sqrt(2 * z / R_e)
    theta_boundary = (np.pi / 2) + theta_h
    
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)
    
    C_layer = n_z * (1 / np.sin(delta)) * dz * (2 * np.pi * frequency / c)**4 * (D / 2)**6
    
    # Both methods now just multiply and sum (Method 2 rotated inside the function)
    Q_detected_M1 += C_layer * np.sum(M21_method_1 * T_in * np.sin(theta_grid)) * dtheta * dphi
    Q_detected_M2 += C_layer * np.sum(M21_method_2 * T_in * np.sin(theta_grid)) * dtheta * dphi

print("=== RESULTS ===")
print(f"Q Polarization (Method 1 - Earth Frame):   {Q_detected_M1:.6e} K")
print(f"Q Polarization (Method 2 - Textbook):      {Q_detected_M2:.6e} K")
print(f"Absolute Difference:                       {abs(Q_detected_M1 - Q_detected_M2):.6e} K")

=== RESULTS ===
Q Polarization (Method 1 - Earth Frame):   2.618889e-05 K
Q Polarization (Method 2 - Textbook):      2.618889e-05 K
Absolute Difference:                       3.388132e-20 K


In [31]:
print("grid1:", M21_method_1)  # Print the raw M21 grids for visual inspection
print("grid2:", M21_method_2)  # Print the raw M21 grids for visual inspection

grid1: [[-0.00249674 -0.00249674 -0.00249674 ... -0.00249674 -0.00249674
  -0.00249674]
 [-0.00233838 -0.00233867 -0.00233954 ... -0.00233954 -0.00233867
  -0.00233838]
 [-0.00218067 -0.00218118 -0.00218273 ... -0.00218273 -0.00218118
  -0.00218067]
 ...
 [-0.0028128  -0.00281205 -0.00280978 ... -0.00280978 -0.00281205
  -0.0028128 ]
 [-0.00265509 -0.00265474 -0.00265369 ... -0.00265369 -0.00265474
  -0.00265509]
 [-0.00249674 -0.00249674 -0.00249674 ... -0.00249674 -0.00249674
  -0.00249674]]
grid2: [[-0.00249674 -0.00249674 -0.00249674 ... -0.00249674 -0.00249674
  -0.00249674]
 [-0.00233838 -0.00233867 -0.00233954 ... -0.00233954 -0.00233867
  -0.00233838]
 [-0.00218067 -0.00218118 -0.00218273 ... -0.00218273 -0.00218118
  -0.00218067]
 ...
 [-0.0028128  -0.00281205 -0.00280978 ... -0.00280978 -0.00281205
  -0.0028128 ]
 [-0.00265509 -0.00265474 -0.00265369 ... -0.00265369 -0.00265474
  -0.00265509]
 [-0.00249674 -0.00249674 -0.00249674 ... -0.00249674 -0.00249674
  -0.00249674]]


In [12]:
#1. Get polarizabilities for your actual shape (e.g., a plate with aspect ratio 0.1)
alpha_h_plate, alpha_v_plate = compute_polarizability(frequency, aspect_ratio=aspect_ratio)

# 2. Get polarizabilities for a perfect sphere (aspect ratio 1.0)
alpha_sphere, _ = compute_polarizability(frequency, aspect_ratio=1.0)

# 3. Compute the Phase Matrices for both
M21_plate = compute_earth_frame_phase_matrix(alpha_h_plate, alpha_v_plate, theta_grid, phi_grid, delta)
M21_sphere = compute_earth_frame_phase_matrix(alpha_sphere, alpha_sphere, theta_grid, phi_grid, delta)

# 4. ISOLATE THE SHAPE EFFECT
# By subtracting the sphere's matrix, you remove the baseline 
# geometric scattering polarization.
M21_shape_induced = M21_plate - M21_sphere

# 5. Integrate as normal using ONLY the shape-induced matrix
Q_shape_induced = 0.0

for z in altitude_layers:
    # ... (calculate theta_h and T_in just like before) ...
    theta_h = np.sqrt(2 * z / R_e)
    theta_boundary = (np.pi / 2) + theta_h
    
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)

    C_layer = n_z * (1 / np.sin(delta)) * dz * (2 * np.pi * frequency / c)**4 * (D / 2)**6
    
    # Use the isolated matrix!
    dQ_layer = C_layer * np.sum(M21_shape_induced * T_in * np.sin(theta_grid)) * dtheta * dphi
    Q_shape_induced += dQ_layer

print(f"Polarization strictly due to particle shape: {Q_shape_induced:.4e} K")

Polarization strictly due to particle shape: 0.0000e+00 K


In [24]:

c = 3e8                 
frequency = 90e9        
R_e = 6371e3            
T_ground = 270.0        
T_sky_zenith = 0    

delta = np.radians(45)  
aspect_ratio = 0.1      # Plates
D = 100e-6              
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5                              

N_theta, N_phi = 100, 100
theta = np.linspace(0, np.pi, N_theta)
phi = np.linspace(0, 2*np.pi, N_phi)
dtheta, dphi = theta[1] - theta[0], phi[1] - phi[0]
theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')


#1. Get polarizabilities for your actual shape (e.g., a plate with aspect ratio 0.1)
alpha_h_plate, alpha_v_plate = compute_polarizability(frequency, aspect_ratio=aspect_ratio)

# 2. Get polarizabilities for a perfect sphere (aspect ratio 1.0)
alpha_sphere, _ = compute_polarizability(frequency, aspect_ratio=1.0)

# 3. Compute the Phase Matrices for both
M21_plate = compute_earth_frame_phase_matrix(alpha_h_plate, alpha_v_plate, theta_grid, phi_grid, delta)
M21_sphere = compute_earth_frame_phase_matrix(alpha_sphere, alpha_sphere, theta_grid, phi_grid, delta)

# 4. ISOLATE THE SHAPE EFFECT
# By subtracting the sphere's matrix, you remove the baseline 
# geometric scattering polarization.
M21_shape_induced = M21_plate - M21_sphere

# 5. Integrate as normal using ONLY the shape-induced matrix
Q_shape_induced = 0.0

for z in altitude_layers:
    # ... (calculate theta_h and T_in just like before) ...
    theta_h = np.sqrt(2 * z / R_e)
    theta_boundary = (np.pi / 2) #+ theta_h
    
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)

    C_layer = n_z * (1 / np.sin(delta)) * dz * (2 * np.pi * frequency / c)**4 * (D / 2)**6
    
    # Use the isolated matrix!
    dQ_layer = C_layer * np.sum(M21_shape_induced * T_in * np.sin(theta_grid)) * dtheta * dphi
    Q_shape_induced += dQ_layer

print(f"Polarization strictly due to particle shape: {Q_shape_induced:.4e} K")

Polarization strictly due to particle shape: -3.0095e-04 K


In [40]:

import numpy as np

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) for the SYMMETRY axis 
    of a spheroid based on its aspect ratio.
    """
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    mask_prolate = (m > 1.0)  # Columns
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    mask_oblate = (m < 1.0)   # Plates
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta


import numpy as np

def compute_polarizability(frequency, aspect_ratio=1.0):
    """
    Computes the complex polarizabilities (alpha_h, alpha_v) for ice crystals.
    Constructs an effective complex alpha_h for columns to mathematically respect 
    the incoherent azimuthal averaging of the absolute squares.
    """
    # 1. Complex relative permittivity of ice
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # 2. Get intrinsic properties
    delta = compute_depolarization_factor(aspect_ratio)
    A_par = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta))
    A_perp = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta) / 2.0))
    
    # 3. Map to Earth-frame
    if aspect_ratio == 1.0:
        # SPHERE
        alpha_v = A_par
        alpha_h = A_par
        
    elif aspect_ratio < 1.0:
        # PLATES (Horizontal plane is pure transverse)
        alpha_v = A_par
        alpha_h = A_perp
        
    else:
        # COLUMNS (Helicopter Spin)
        alpha_v = A_perp
        
        # Target 1: Averaged Intensity (Scattering)
        target_abs2_h = (np.abs(A_par)**2 + np.abs(A_perp)**2) / 2.0
        
        # Target 2: Averaged Absorption (Emission)
        target_imag_h = (np.imag(A_par) + np.imag(A_perp)) / 2.0
        
        # Construct the effective complex alpha_h
        real_h = np.sqrt(np.maximum(0, target_abs2_h - target_imag_h**2))
        alpha_h = real_h + 1j * target_imag_h
        
    return alpha_h, alpha_v

# ====================================================================
# METHOD 1: DIRECT EARTH FRAME (Fast & Elegant)
# ====================================================================
def compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    abs_h2 = np.abs(alpha_h)**2
    abs_v2 = np.abs(alpha_v)**2
    
    I_H = abs_h2 * (1.0 - (np.sin(theta_grid) * np.sin(phi_grid))**2)
    
    ray_proj = alpha_v * np.cos(delta) * np.cos(theta_grid) - \
               alpha_h * np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    I_V = (abs_h2 * np.sin(delta)**2 + abs_v2 * np.cos(delta)**2) - np.abs(ray_proj)**2
    
    M21_earth = 0.5 * (I_V - I_H)
    return M21_earth

# ====================================================================
# METHOD 2: TEXTBOOK SCATTERING PLANE (Corrected Vector Math)
# ====================================================================
def compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    
    # 1. Define Propagation Vectors (k_i and k_s)
    s_i = np.stack([-np.sin(theta_grid) * np.cos(phi_grid), 
                    -np.sin(theta_grid) * np.sin(phi_grid), 
                    -np.cos(theta_grid)], axis=-1)
    
    # Corrected Code:
    s_s = np.array([-np.cos(delta), 0.0, -np.sin(delta)])
    s_s = np.broadcast_to(s_s, s_i.shape)
    
    # 2. Build the 2D Scattering Plane Unit Vectors
    n_scat = np.cross(s_i, s_s, axis=-1)
    norm = np.linalg.norm(n_scat, axis=-1, keepdims=True)
    norm = np.where(norm < 1e-12, 1e-12, norm) 
    e_perp = n_scat / norm
    
    # CORRECTED: Strict Right-Hand Rule (e_par = e_perp x k)
    e_par_i = np.cross(e_perp, s_i, axis=-1)
    e_par_s = np.cross(e_perp, s_s, axis=-1)
    
    # 3. Apply the Earth-locked Ice Crystal Tensor
    def apply_tensor(v):
        return np.stack([alpha_h * v[..., 0], alpha_h * v[..., 1], alpha_v * v[..., 2]], axis=-1)
    
    # 4. Calculate Bohren & Huffman Amplitude Matrix Elements
    S1 = np.sum(e_perp * apply_tensor(e_perp), axis=-1)
    S2 = np.sum(e_par_s * apply_tensor(e_par_i), axis=-1)
    S3 = np.sum(e_par_s * apply_tensor(e_perp), axis=-1)
    S4 = np.sum(e_perp * apply_tensor(e_par_i), axis=-1)
    
    # 5. Stokes Phase Matrix inside the local scattering plane (F)
    F21 = 0.5 * (np.abs(S2)**2 + np.abs(S3)**2 - np.abs(S1)**2 - np.abs(S4)**2)
    F31 = np.real(S2 * np.conj(S4) + S3 * np.conj(S1))
    
    # 6. EXACT Rotation: Bypass spherical trig and use exact 3D Dot Products
    e_v = np.array([-np.sin(delta), 0.0, np.cos(delta)])
    e_v = np.broadcast_to(e_v, s_i.shape)
    
    # Project the telescope vertical sensor onto the scattering plane basis
    cos_psi = np.sum(e_v * e_par_s, axis=-1)
    sin_psi = -np.sum(e_v * e_perp, axis=-1)
    
    # Apply double-angle formulas for the Mueller rotation matrix R(psi)
    cos_2psi = cos_psi**2 - sin_psi**2
    sin_2psi = 2 * sin_psi * cos_psi
    
    # Corrected Code:
    M21_rotated = F21 * cos_2psi - F31 * sin_2psi
    
    return M21_rotated

# ====================================================================
# EXECUTE SIMULATION
# ====================================================================
c = 3e8                 
frequency = 90e9        
R_e = 6371e3            
T_ground = 270.0        
T_sky_zenith = 0 #10.0     

delta = np.radians(45)  
aspect_ratio = 0.1  # Plates
D = 100e-6              
dz = 100.0                             
altitude_layers = np.arange(6000, 8000, dz) 
n_z = 1e5                              

N_theta, N_phi = 100, 100
theta = np.linspace(0, np.pi, N_theta)
phi = np.linspace(0, 2*np.pi, N_phi)
dtheta, dphi = theta[1] - theta[0], phi[1] - phi[0]
theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')

# Pre-compute Physics
alpha_h, alpha_v = compute_polarizability(frequency, aspect_ratio)

# Generate the Matrix grids using BOTH methods
M21_method_1 = compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)
M21_method_2 = compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta)

# Integrate
Q_detected_M1 = 0.0
Q_detected_M2 = 0.0

for z in altitude_layers:
    theta_h = np.sqrt(2 * z / R_e)
    theta_boundary = (np.pi / 2) #+ theta_h
    
    T_sky_profile = T_sky_zenith / np.cos(np.clip(theta_grid, 0, np.pi/2 - 0.01))
    T_in = np.where(theta_grid >= theta_boundary, T_ground, T_sky_profile)
    
    # Corrected Code:
    V_squared = (16.0 / 9.0) * np.pi**2 * (D / 2.0)**6
    k_4 = (2 * np.pi * frequency / c)**4
    C_layer = n_z * (1 / np.sin(delta)) * dz * k_4 * V_squared
    
    # Both methods now just multiply and sum (Method 2 rotated inside the function)
    Q_detected_M1 += C_layer * np.sum(M21_method_1 * T_in * np.sin(theta_grid)) * dtheta * dphi
    Q_detected_M2 += C_layer * np.sum(M21_method_2 * T_in * np.sin(theta_grid)) * dtheta * dphi

print("=== RESULTS ===")
print(f"Q Polarization (Method 1 - Earth Frame):   {Q_detected_M1:.6e} K")
print(f"Q Polarization (Method 2 - Textbook):      {Q_detected_M2:.6e} K")
print(f"Absolute Difference:                       {abs(Q_detected_M1 - Q_detected_M2):.6e} K")

=== RESULTS ===
Q Polarization (Method 1 - Earth Frame):   -5.349985e-03 K
Q Polarization (Method 2 - Textbook):      -5.349985e-03 K
Absolute Difference:                       0.000000e+00 K


Not vectorized along aspect ratio below

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants
from atm_tools import alpha_specific_function

pi = constants.pi 
c = constants.c


#The first part is for SPT 

def create_constant_ice_layer(altitudes, layer_bottom, layer_top, ice_density):
    """
    Create a constant ice layer with specified density between given altitudes.

    Parameters:
    altitudes (numpy array): Array of altitude values in meters.
    layer_bottom (float): Bottom altitude of the ice layer in meters.
    layer_top (float): Top altitude of the ice layer in meters.
    ice_density (float): Density of ice particles in particles/m^3.

    Returns:
    numpy array: Array of particle densities corresponding to the input altitudes.
    """
    n = np.zeros_like(altitudes)  # Initialize an array of zeros
    in_layer = (altitudes >= layer_bottom) & (altitudes <= layer_top)  # Find indices within the layer
    n[in_layer] = ice_density  # Set density for those indices
    return n

def sigma_scattering(frequency, volume, A):
    """
    Calculate the scattering cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The scattering cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_sca = w**4*volume**2*np.abs(A)**2/(6*pi*c**4)
    
    return sigma_sca

def sigma_absorption(frequency, volume, A):
    """
    Calculate the absorption cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The absorption cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_abs = w*volume*np.imag(A)/c

    return sigma_abs

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) along the symmetry axis
    for a spheroid given its aspect ratio.
    
    Parameters:
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
                          m = 1.0 (sphere), m > 1.0 (column), m < 1.0 (plate).
    
    Returns:
    float: Depolarization factor Delta.
    """
    m = aspect_ratio
    
    if m == 1.0:
        # Sphere
        return 1.0 / 3.0
        
    elif m > 1.0:
        # Prolate spheroid (Column)
        e = np.sqrt(1.0 - (1.0 / m)**2)
        delta = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        return delta
        
    else:
        # Oblate spheroid (Plate)
        e = np.sqrt(1.0 - m**2)
        delta = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        return delta

def compute_intrinsic_polarizabilities(frequency, aspect_ratio):
    """Computes the inherent polarizabilities (A_par, A_perp) relative to the particle's symmetry axis."""
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    if aspect_ratio == 1.0:
        A_par = 3 * (eps - 1) / (eps + 2)
        return A_par, A_par
    
    delta = compute_depolarization_factor(aspect_ratio)
    A_par = (eps - 1) / (1 + (eps - 1) * delta)
    A_perp = (eps - 1) / (1 + (eps - 1) * (1 - delta) / 2.0)
    
    return A_par, A_perp


def compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param='I'):
    """
    Projects the polarizabilities onto the telescope based on aerodynamics and elevation.
    Correctly handles the incoherent azimuthal averaging for spinning columns.
    """
    # 1. Get intrinsic complex polarizabilities
    A_par, A_perp = compute_intrinsic_polarizabilities(frequency, aspect_ratio)
    
    # 2. Pre-calculate the physical intensities (abs^2) and absorption (imag)
    abs2_par = np.abs(A_par)**2
    abs2_perp = np.abs(A_perp)**2
    
    imag_par = np.imag(A_par)
    imag_perp = np.imag(A_perp)
    
    # 3. Map to Earth-frame based on aerodynamics (Incoherent Averaging)
    if aspect_ratio == 1.0:
        # Sphere
        abs2_v, abs2_h = abs2_par, abs2_par
        imag_v, imag_h = imag_par, imag_par
        
    elif aspect_ratio < 1.0:
        # Plates (Fall flat, symmetry axis is vertical)
        abs2_v, abs2_h = abs2_par, abs2_perp
        imag_v, imag_h = imag_par, imag_perp
        
    else:
        # Columns (Fall flat, spin randomly in azimuth)
        # Vertical sees pure transverse. Horizontal is the average of parallel and transverse.
        abs2_v = abs2_perp
        abs2_h = (abs2_par + abs2_perp) / 2.0
        
        imag_v = imag_perp
        imag_h = (imag_par + imag_perp) / 2.0
        
    # 4. Project onto Telescope Sensors based on Stokes parameter
    eps_rad = np.radians(elevation)
    
    if stokes_param == 'I':
        eff_abs2 = 0.5 * (abs2_v * np.cos(eps_rad)**2 + abs2_h * (1.0 + np.sin(eps_rad)**2))
        eff_imag = 0.5 * (imag_v * np.cos(eps_rad)**2 + imag_h * (1.0 + np.sin(eps_rad)**2))
        
    elif stokes_param == 'Q':
        eff_abs2 = 0.5 * (abs2_v - abs2_h) * np.cos(eps_rad)**2
        eff_imag = 0.5 * (imag_v - imag_h) * np.cos(eps_rad)**2
        
    else:
        raise ValueError("stokes_param must be 'I' or 'Q'")

    return eff_abs2, eff_imag

def polarization_fraction_2(frequency, elevation_deg, aspect_ratio=0.5, process='scattering'):
    """
    Computes the polarization fraction (Q/I) directly from the effective polarizabilities,
    bypassing the intermediate gamma ratio calculation.
    
    Parameters:
    frequency (float or array): Frequency in Hz.
    elevation_deg (float or array): Telescope elevation angle in degrees.
    aspect_ratio (float): Ratio of symmetry axis to transverse axis. Defaults to 0.5 (plate).
    process (str): 'scattering' or 'emission'.
    
    Returns:
    float or array: The polarization fraction p_gamma.
    """
    
    # 1. Get the effective polarizabilities for Total Intensity (I)
    eff_abs2_I, eff_imag_I = compute_effective_polarizability(
        frequency, aspect_ratio, elevation_deg, stokes_param='I'
    )
    
    # 2. Get the effective polarizabilities for Linear Polarization (Q)
    eff_abs2_Q, eff_imag_Q = compute_effective_polarizability(
        frequency, aspect_ratio, elevation_deg, stokes_param='Q'
    )
    
    # 3. Calculate the fraction depending on the physical process
    if process == 'scattering':
        # Scattering scales with the absolute square of the polarizability
        p_frac = eff_abs2_Q / eff_abs2_I
        
    elif process == 'emission':
        # Thermal emission scales with the imaginary part of the polarizability (absorption)
        p_frac = eff_imag_Q / eff_imag_I
        
    else:
        raise ValueError("Process must be 'scattering' or 'emission'.")
        
    return p_frac



def compute_T_RJ_ice2(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, radius_eq, aspect_ratio=1.0, process='total', stokes_param='I'):
    """
    Computes the Rayleigh-Jeans brightness temperature contribution from ice crystals 
    along a line of sight for a specific Stokes parameter (I or Q).
    
    Parameters:
    frequency (array-like): Array of frequencies in Hz. (Nf)
    altitudes (array-like): Array of altitudes in meters. (Nz)
    Temperature (array-like): Array of atmospheric temperatures in K. (Nz)
    Pressure (array-like): Array of atmospheric pressures in hPa. (Nz)
    P_water (array-like): Array of water vapor partial pressures in hPa. (Nz)
    elevation (float): Elevation angle of the telescope in degrees.
    ice_density (array-like): Array of ice water content (in particles/m^3). (Nz, Na)
    radius_eq (array-like): Array of equivalent radii of the ice crystals in meters. (Na)
    aspect_ratio (float or array): Ratio of symmetry axis to transverse axis.
    process (str): 'scattering', 'emission', or 'total'.
    stokes_param (str): 'I' (Total Intensity) or 'Q' (Linear Polarization).

    Returns:
    ndarray: The Rayleigh-Jeans brightness temperature array (shape: Nf, Na).
    """
    c = constants.c
    
    # 1. Geometry and airmass
    zenith_angle = 90.0 - elevation
    m = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    dz = np.diff(altitudes) # Shape: (Nz-1,)

    # Particle volumes
    V = (4.0 / 3.0) * np.pi * radius_eq**3 # Shape: (Na,)
    
    # 2. Get Effective Cross-Section Multipliers
    eff_abs2_sca_req, eff_imag_abs_req = compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param=stokes_param)
    eff_abs2_sca_I, eff_imag_abs_I = compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param='I')

    # 3. Calculate Ice Cross-Sections and Alphas
    k_4 = (2 * np.pi * frequency[:, None] / c)**4
    k_1 = (2 * np.pi * frequency[:, None] / c)
    geom_factor = 1.0 / (6.0 * np.pi)

    sigma_sca_req = geom_factor * k_4 * (V[None, :]**2) * eff_abs2_sca_req[:, None]
    sigma_sca_I   = geom_factor * k_4 * (V[None, :]**2) * eff_abs2_sca_I[:, None]

    sigma_abs_req = k_1 * V[None, :] * eff_imag_abs_req[:, None]
    sigma_abs_I   = k_1 * V[None, :] * eff_imag_abs_I[:, None]

    alpha_sca_req = ice_density[:, None, :] * sigma_sca_req[None, :, :] 
    alpha_sca_I   = ice_density[:, None, :] * sigma_sca_I[None, :, :] 
    alpha_abs_req = ice_density[:, None, :] * sigma_abs_req[None, :, :] 
    alpha_abs_I   = ice_density[:, None, :] * sigma_abs_I[None, :, :] 

    # Midpoint averages for ice layers
    alpha_sca_req_mid = (alpha_sca_req[:-1, :, :] + alpha_sca_req[1:, :, :]) / 2.0
    alpha_sca_I_mid   = (alpha_sca_I[:-1, :, :] + alpha_sca_I[1:, :, :]) / 2.0
    alpha_abs_req_mid = (alpha_abs_req[:-1, :, :] + alpha_abs_req[1:, :, :]) / 2.0
    alpha_abs_I_mid   = (alpha_abs_I[:-1, :, :] + alpha_abs_I[1:, :, :]) / 2.0

    # Optical thicknesses for ice: Shape (Nz-1, Nf, Na)
    d_tau_sca_req = alpha_sca_req_mid * dz[:, None, None]
    d_tau_sca_I   = alpha_sca_I_mid * dz[:, None, None]
    d_tau_abs_req = alpha_abs_req_mid * dz[:, None, None]
    d_tau_abs_I   = alpha_abs_I_mid * dz[:, None, None]

    # Total Ice Optical Depth (Intensity) for cumulative attenuation
    d_tau_ext_I = d_tau_sca_I + d_tau_abs_I
    
    # Cumulative ice attenuation from below
    tau_below_ice = np.cumsum(d_tau_ext_I, axis=0)
    tau_below_ice = np.insert(tau_below_ice[:-1, :, :], 0, 0, axis=0) 

    # 4. Atmospheric (Gas) Optical Depths
    alpha_atm = alpha_specific_function(altitudes, frequency, Temperature, Pressure, P_water) # Shape: (Nz, Nf)
    alpha_atm_mid = (alpha_atm[:-1, :] + alpha_atm[1:, :]) / 2.0
    
    d_tau_atm = alpha_atm_mid * dz[:, None] # Shape: (Nz-1, Nf)
    tau_below_atm = np.cumsum(d_tau_atm, axis=0)
    tau_below_atm = np.insert(tau_below_atm[:-1, :], 0, 0, axis=0) 

    # Midpoint Thermodynamic Temperatures
    T_mid = (Temperature[:-1] + Temperature[1:]) / 2.0 # Shape: (Nz-1,)

    # 5. The Universal Attenuation Factor to reach the telescope
    # Broadcast tau_below_atm to match ice dimensions (Nz-1, Nf, Na)
    attenuation = np.exp(-(tau_below_ice + tau_below_atm[:, :, None]) * m)

    # 6. Radiative Transfer Integration
    T_layers_sca = 0.0
    T_layers_abs = 0.0

    if process in ['scattering', 'total']:
        # Source is ground temperature (approximated half-hemisphere)
        T_source_sca = Temperature[0] / 2.0 
        
        p_gamma_sca = np.divide(d_tau_sca_req, d_tau_sca_I, out=np.zeros_like(d_tau_sca_I), where=(d_tau_sca_I != 0))
        
        # Radiative Transfer: Source * Generation * Attenuation
        T_layers_sca_I = T_source_sca * (1 - np.exp(-d_tau_sca_I * m))
        T_layers_sca = p_gamma_sca * T_layers_sca_I * attenuation

    if process in ['emission', 'total']:
        T_source_abs = T_mid[:, None, None]
        
        p_gamma_abs = np.divide(d_tau_abs_req, d_tau_abs_I, out=np.zeros_like(d_tau_abs_I), where=(d_tau_abs_I != 0))
        
        # Radiative Transfer: Source * Generation * Attenuation
        T_layers_abs_I = T_source_abs * (1 - np.exp(-d_tau_abs_I * m))
        T_layers_abs = p_gamma_abs * T_layers_abs_I * attenuation

    # 7. Selection and Summation
    if process == 'scattering':
        T_layers = T_layers_sca
    elif process == 'emission':
        T_layers = T_layers_abs
    else: 
        T_layers = T_layers_sca + T_layers_abs

    T_sky = np.sum(T_layers, axis=0) 
    
    return T_sky




#Let's go over the CLASS code

#The brightness temperature seen by an ice crystal

import numpy as np

def compute_T_in_1D(theta_grid, z_part, altitudes, T_phys_profile, T_ground, tau_z_profile):
    """
    Computes the full angular profile of incoming brightness temperature hitting a particle.
    
    Parameters:
    theta_grid     : ND array of zenith angles in radians (e.g., shape (1, N_theta, N_phi))
    z_part         : Scalar, altitude of the active layer/particle in meters
    altitudes      : 1D array of atmospheric profile altitudes in meters (ascending)
    T_phys_profile : 1D array of thermodynamic temperatures corresponding to altitudes
    T_ground       : Scalar, physical temperature of the ground
    tau_z_profile  : 1D array, cumulative optical depth from the ground up to 'altitudes'
    """
    R_e = 6371e3  # Earth radius in meters
    
    # ---------------------------------------------------------
    # 1. EXTRACT LOCAL LAYER PROPERTIES
    # ---------------------------------------------------------
    # Interpolate to find exactly what is happening at z_part
    tau_below = np.interp(z_part, altitudes, tau_z_profile)
    tau_total = tau_z_profile[-1]
    tau_above = tau_total - tau_below
    
    # Estimate effective temperatures of the atmosphere above and below the particle
    mask_above = altitudes >= z_part
    mask_below = altitudes <= z_part
    T_atm_above = np.mean(T_phys_profile[mask_above]) if np.any(mask_above) else T_phys_profile[-1]
    T_atm_below = np.mean(T_phys_profile[mask_below]) if np.any(mask_below) else T_ground

    # ---------------------------------------------------------
    # 2. DEFINE THE ANGULAR BOUNDARIES
    # ---------------------------------------------------------
    theta_h = np.sqrt(2 * z_part / R_e)
    bound_sky = np.pi / 2.0
    bound_limb = (np.pi / 2.0) + theta_h

    # ---------------------------------------------------------
    # 3. ZONE 1: THE UPPER SKY (theta < pi/2)
    # ---------------------------------------------------------
    # Kasten-Young empirical airmass
    theta_deg = np.degrees(theta_grid)
    theta_deg_sky = np.clip(theta_deg, 0, 90.0) 
    m_sky = 1.0 / (np.cos(theta_grid) + 0.50572 * (96.07995 - theta_deg_sky)**(-1.6364))
    
    T_sky = T_atm_above * (1.0 - np.exp(-tau_above * m_sky))

    # ---------------------------------------------------------
    # 4. ZONE 2: THE TANGENT LIMB (pi/2 <= theta <= pi/2 + theta_h)
    # ---------------------------------------------------------
    # Calculate the minimum grazing altitude of the ray
    z_min_grid = (R_e + z_part) * np.sin(theta_grid) - R_e
    
    # Interpolate physical temperature at the grazing altitude
    T_limb_flat = np.interp(z_min_grid.flatten(), altitudes, T_phys_profile)
    T_limb = T_limb_flat.reshape(theta_grid.shape)

    # ---------------------------------------------------------
    # 5. ZONE 3: THE GROUND (theta > pi/2 + theta_h)
    # ---------------------------------------------------------
    # For looking down, we use the downward secant law: 1 / cos(pi - theta)
    # We clip the angle slightly away from exactly 90 deg to prevent division by zero
    theta_down = np.clip(theta_grid, bound_limb + 1e-4, np.pi)
    m_down = 1.0 / np.cos(np.pi - theta_down)
    
    # Emission from the ground (attenuated) + emission from the air below the particle
    T_ground_zone = T_ground * np.exp(-tau_below * m_down) + \
                    T_atm_below * (1.0 - np.exp(-tau_below * m_down))

    # ---------------------------------------------------------
    # 6. STITCH THE ZONES TOGETHER
    # ---------------------------------------------------------
    conditions = [
        theta_grid < bound_sky,
        (theta_grid >= bound_sky) & (theta_grid <= bound_limb),
        theta_grid > bound_limb
    ]
    
    choices = [T_sky, T_limb, T_ground_zone]
    
    T_in = np.select(conditions, choices)
    
    return T_in

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) for the SYMMETRY axis 
    of a spheroid based on its aspect ratio.
    """
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    mask_prolate = (m > 1.0)  # Columns
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    mask_oblate = (m < 1.0)   # Plates
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta


import numpy as np

def compute_polarizability(frequency, aspect_ratio=1.0):
    """
    Computes the complex polarizabilities (alpha_h, alpha_v) for ice crystals.
    Constructs an effective complex alpha_h for columns to mathematically respect 
    the incoherent azimuthal averaging of the absolute squares.
    """
    # 1. Complex relative permittivity of ice
    eps_prime = 3.16
    eps_double_prime = 8e-3 * (frequency / 150e9) 
    eps = eps_prime + 1j * eps_double_prime
    
    # 2. Get intrinsic properties
    delta = compute_depolarization_factor(aspect_ratio)
    A_par = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta))
    A_perp = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta) / 2.0))
    
    # 3. Map to Earth-frame
    if aspect_ratio == 1.0:
        # SPHERE
        alpha_v = A_par
        alpha_h = A_par
        
    elif aspect_ratio < 1.0:
        # PLATES (Horizontal plane is pure transverse)
        alpha_v = A_par
        alpha_h = A_perp
        
    else:
        # COLUMNS (Helicopter Spin)
        alpha_v = A_perp
        
        # Target 1: Averaged Intensity (Scattering)
        target_abs2_h = (np.abs(A_par)**2 + np.abs(A_perp)**2) / 2.0
        
        # Target 2: Averaged Absorption (Emission)
        target_imag_h = (np.imag(A_par) + np.imag(A_perp)) / 2.0
        
        # Construct the effective complex alpha_h
        real_h = np.sqrt(np.maximum(0, target_abs2_h - target_imag_h**2))
        alpha_h = real_h + 1j * target_imag_h
        
    return alpha_h, alpha_v

import numpy as np

# ====================================================================
# METHOD 1: DIRECT EARTH FRAME (Fast & Elegant)
# ====================================================================
def compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    abs_h2 = np.abs(alpha_h)**2
    abs_v2 = np.abs(alpha_v)**2
    
    # Scattered intensity projected onto the Horizontal telescope axis
    I_H = abs_h2 * (1.0 - (np.sin(theta_grid) * np.sin(phi_grid))**2)
    
    # Scattered intensity projected onto the Vertical telescope axis
    ray_proj = alpha_v * np.cos(delta) * np.cos(theta_grid) - \
               alpha_h * np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    I_V = (abs_h2 * np.sin(delta)**2 + abs_v2 * np.cos(delta)**2) - np.abs(ray_proj)**2
    
    # Calculate M11 (Total Intensity) and M21 (Linear Polarization Q)
    M11_earth = 0.5 * (I_V + I_H)
    M21_earth = 0.5 * (I_V - I_H)
    
    return M11_earth, M21_earth


# ====================================================================
# METHOD 2: TEXTBOOK SCATTERING PLANE (Corrected Vector Math)
# ====================================================================
def compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    
    # 1. Define Propagation Vectors (k_i and k_s)
    s_i = np.stack([-np.sin(theta_grid) * np.cos(phi_grid), 
                    -np.sin(theta_grid) * np.sin(phi_grid), 
                    -np.cos(theta_grid)], axis=-1)
    
    # Propagating down to the telescope
    s_s = np.array([-np.cos(delta), 0.0, -np.sin(delta)])
    s_s = np.broadcast_to(s_s, s_i.shape)
    
    # 2. Build the 2D Scattering Plane Unit Vectors
    n_scat = np.cross(s_i, s_s, axis=-1)
    norm = np.linalg.norm(n_scat, axis=-1, keepdims=True)
    norm = np.where(norm < 1e-12, 1e-12, norm) 
    e_perp = n_scat / norm
    
    # Strict Right-Hand Rule (e_par = e_perp x k)
    e_par_i = np.cross(e_perp, s_i, axis=-1)
    e_par_s = np.cross(e_perp, s_s, axis=-1)
    
    # 3. Apply the Earth-locked Ice Crystal Tensor
    def apply_tensor(v):
        return np.stack([alpha_h * v[..., 0], alpha_h * v[..., 1], alpha_v * v[..., 2]], axis=-1)
    
    # 4. Calculate Bohren & Huffman Amplitude Matrix Elements
    S1 = np.sum(e_perp * apply_tensor(e_perp), axis=-1)
    S2 = np.sum(e_par_s * apply_tensor(e_par_i), axis=-1)
    S3 = np.sum(e_par_s * apply_tensor(e_perp), axis=-1)
    S4 = np.sum(e_perp * apply_tensor(e_par_i), axis=-1)
    
    # 5. Stokes Phase Matrix inside the local scattering plane (F)
    # Added F11 which sums all power states
    F11 = 0.5 * (np.abs(S1)**2 + np.abs(S2)**2 + np.abs(S3)**2 + np.abs(S4)**2)
    F21 = 0.5 * (np.abs(S2)**2 + np.abs(S3)**2 - np.abs(S1)**2 - np.abs(S4)**2)
    F31 = np.real(S2 * np.conj(S4) + S3 * np.conj(S1))
    
    # 6. EXACT Rotation: Bypass spherical trig and use exact 3D Dot Products
    e_v = np.array([-np.sin(delta), 0.0, np.cos(delta)])
    e_v = np.broadcast_to(e_v, s_i.shape)
    
    # Project the telescope vertical sensor onto the scattering plane basis
    cos_psi = np.sum(e_v * e_par_s, axis=-1)
    sin_psi = -np.sum(e_v * e_perp, axis=-1)
    
    # Apply double-angle formulas for the Mueller rotation matrix R(psi)
    cos_2psi = cos_psi**2 - sin_psi**2
    sin_2psi = 2 * sin_psi * cos_psi
    
    # M11 bypasses rotation entirely. M21 is rotated into the telescope frame.
    M11_rotated = F11 
    M21_rotated = F21 * cos_2psi - F31 * sin_2psi
    
    return M11_rotated, M21_rotated


import numpy as np
from scipy import constants

# ====================================================================
# HELPER: VECTORIZED 3D SKY TEMPERATURE
# ====================================================================
"""
def compute_T_in(theta_grid, z_layers, altitudes, T_phys_profile, T_ground, tau_above_atm, tau_below_atm):
    
    #Computes the incoming brightness temperature hitting the ice crystal from all angles.
    #Fully vectorized across layers (Nz), frequencies (Nf), and angles (N_theta, N_phi).
    
    R_e = 6371e3
    
    # Expand physical arrays for 4D broadcasting: (Nz, Nf, N_theta, N_phi)
    # tg shape: (1, 1, N_theta, N_phi)
    tg = theta_grid[None, None, :, :] 
    z_exp = z_layers[:, None, None, None]
    T_mid = ((T_phys_profile[:-1] + T_phys_profile[1:]) / 2.0)[:, None, None, None]
    
    tau_above = tau_above_atm[:, :, None, None]
    tau_below = tau_below_atm[:, :, None, None]
    
    # Geometric Boundaries
    theta_h = np.sqrt(2 * z_exp / R_e)
    bound_sky = np.pi / 2
    bound_limb = (np.pi / 2) + theta_h
    
    # 1. SKY REGIME (Theta < 90 deg)
    # Using Kasten-Young airmass, clipped exactly at 90 deg
    theta_deg = np.clip(np.degrees(tg), 0, 90.0)
    m_sky = 1.0 / (np.cos(tg) + 0.50572 * (96.07995 - theta_deg)**(-1.6364))
    T_sky = T_mid * (1.0 - np.exp(-tau_above * m_sky))
    
    # 2. LIMB REGIME (90 <= Theta <= 90 + theta_h)
    # Saturated tangent ray takes the physical temperature of its minimum altitude
    z_min = (R_e + z_exp) * np.sin(tg) - R_e
    T_limb_flat = np.interp(z_min.flatten(), altitudes, T_phys_profile)
    T_limb = T_limb_flat.reshape(z_min.shape)
    
    # 3. GROUND REGIME (Theta > 90 + theta_h)
    # Upwelling attenuated ground emission + lower atmosphere emission
    m_gnd = 1.0 / np.maximum(np.cos(np.pi - tg), 0.01) # Avoid div by zero looking horizontally
    T_gnd_attenuated = T_ground * np.exp(-tau_below * m_gnd) + T_mid * (1.0 - np.exp(-tau_below * m_gnd))
    
    # Stitch the regimes together
    conditions = [
        tg < bound_sky,
        (tg >= bound_sky) & (tg <= bound_limb),
        tg > bound_limb
    ]
    choices = [T_sky, T_limb, T_gnd_attenuated]
    
    return np.select(conditions, choices)
"""

def compute_T_in(theta_grid, z_layers, altitudes, T_phys_profile, T_ground, tau_above_atm, tau_below_atm, 
                 consider_earth_curvature=True, consider_atmospheric_emission=True):
    """
    Computes the incoming brightness temperature hitting the ice crystal from all angles.
    Fully vectorized across layers (Nz), frequencies (Nf), and angles (N_theta, N_phi).
    
    Toggles:
    - consider_earth_curvature: If False, assumes a flat Earth (no limb regime).
    - consider_atmospheric_emission: If False, ignores atmospheric emission/attenuation 
                                     (Sky = 0K, Ground = pure T_ground).
    """
    R_e = 6371e3
    
    # Expand physical arrays for 4D broadcasting: (Nz, Nf, N_theta, N_phi)
    tg = theta_grid[None, None, :, :] 
    z_exp = z_layers[:, None, None, None]
    T_mid = ((T_phys_profile[:-1] + T_phys_profile[1:]) / 2.0)[:, None, None, None]
    
    tau_above = tau_above_atm[:, :, None, None]
    tau_below = tau_below_atm[:, :, None, None]
    
    # ==========================================
    # 1. BASE PHYSICS (Atmosphere vs No Atmosphere)
    # ==========================================
    if consider_atmospheric_emission:
        # Sky emission (downwelling)
        theta_deg = np.clip(np.degrees(tg), 0, 90.0)
        m_sky = 1.0 / (np.cos(tg) + 0.50572 * (96.07995 - theta_deg)**(-1.6364))
        T_sky = T_mid * (1.0 - np.exp(-tau_above * m_sky))
        
        # Ground emission (upwelling + atmospheric attenuation/emission)
        m_gnd = 1.0 / np.maximum(np.cos(np.pi - tg), 0.01)
        T_gnd_effective = T_ground * np.exp(-tau_below * m_gnd) + T_mid * (1.0 - np.exp(-tau_below * m_gnd))
    else:
        # No atmosphere: Sky is dark space (0K), Ground is unattenuated
        T_sky = np.zeros_like(tg)
        T_gnd_effective = np.full_like(tg, T_ground)

    # ==========================================
    # 2. GEOMETRIC STITCHING (Curved vs Flat Earth)
    # ==========================================
    if consider_earth_curvature:
        theta_h = np.sqrt(2 * z_exp / R_e)
        bound_sky = np.pi / 2
        bound_limb = (np.pi / 2) + theta_h
        
        if consider_atmospheric_emission:
            # Limb rays saturate to the physical temperature of their lowest atmospheric dip
            z_min = (R_e + z_exp) * np.sin(tg) - R_e
            T_limb_flat = np.interp(z_min.flatten(), altitudes, T_phys_profile)
            T_limb = T_limb_flat.reshape(z_min.shape)
        else:
            # Without atmosphere, limb rays that graze the Earth just hit deep space
            T_limb = np.zeros_like(tg) 
            
        conditions = [
            tg <= bound_sky,
            (tg > bound_sky) & (tg <= bound_limb),
            tg > bound_limb
        ]
        choices = [T_sky, T_limb, T_gnd_effective]
        
    else:
        # Flat Earth Approximation (Only Two Regimes)
        bound_sky = np.pi / 2
        
        conditions = [
            tg <= bound_sky,
            tg > bound_sky
        ]
        choices = [T_sky, T_gnd_effective]
    
    return np.select(conditions, choices)

# ====================================================================
# CLASS INTEGRATION ENGINE (Fully Coupled with Atmosphere)
# ====================================================================

def _compute_T_RJ_ice_CLASS_core(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, 
                                 radius_eq, aspect_ratio, stokes_param, 
                                 consider_earth_curvature, consider_atmospheric_emission, method):
    c = constants.c
    delta = np.radians(elevation)
    
    # 1. Kasten-Young Airmass for Telescope Line of Sight
    zenith_angle = 90.0 - elevation
    m_telescope = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    
    # 2. Standardize Inputs & Grid Alignment
    freq_arr = np.atleast_1d(frequency)
    radius_arr = np.atleast_1d(radius_eq)
    
    ice_dens_arr = np.asarray(ice_density)
    if ice_dens_arr.ndim == 1:
        ice_dens_arr = ice_dens_arr[:, None] 
        
    if ice_dens_arr.shape[0] == len(altitudes):
        ice_dens_arr = (ice_dens_arr[:-1, :] + ice_dens_arr[1:, :]) / 2.0
        
    dz = np.diff(altitudes)
    z_layers = (altitudes[:-1] + altitudes[1:]) / 2.0
    
    # 3. Angular Grid Setup
    N_theta, N_phi = 100, 100
    theta = np.linspace(0, np.pi, N_theta)
    phi = np.linspace(0, 2*np.pi, N_phi, endpoint=False)
    dtheta, dphi = theta[1] - theta[0], phi[1] - phi[0]
    
    theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')
    tg = theta_grid[None, :, :]
    pg = phi_grid[None, :, :]
    
    # 4. Ice Physics Multipliers
    alpha_h, alpha_v = compute_polarizability(freq_arr, aspect_ratio) 
    alpha_h_exp = np.atleast_1d(alpha_h)[:, None, None]
    alpha_v_exp = np.atleast_1d(alpha_v)[:, None, None]
    
    V = (4.0 / 3.0) * np.pi * radius_arr**3
    V_squared = (16.0 / 9.0) * np.pi**2 * (radius_arr)**6
    k_4 = (2 * np.pi * freq_arr / c)**4
    
    if method == 1:
        M11, M21 = compute_earth_frame_phase_matrix(alpha_h_exp, alpha_v_exp, tg, pg, delta)
    else:
        M11, M21 = compute_scattering_plane_phase_matrix(alpha_h_exp, alpha_v_exp, tg, pg, delta)
        
    M = M11 if stokes_param == 'I' else M21
    
    # 5. Atmospheric (Gas) Optical Depths
    alpha_atm = alpha_specific_function(altitudes, freq_arr, Temperature, Pressure, P_water)
    alpha_atm_mid = (alpha_atm[:-1, :] + alpha_atm[1:, :]) / 2.0
    d_tau_atm = alpha_atm_mid * dz[:, None] 
    
    # Attenuation from below (for telescope sightline & ground emission)
    tau_below_atm = np.cumsum(d_tau_atm, axis=0)
    tau_below_atm = np.insert(tau_below_atm[:-1, :], 0, 0, axis=0) 
    
    # Attenuation from above (for sky emission hitting the particle)
    tau_above_atm = np.zeros_like(d_tau_atm)
    if len(d_tau_atm) > 1:
        tau_above_atm[:-1, :] = np.cumsum(d_tau_atm[1:][::-1], axis=0)[::-1]
    
    # The Universal Attenuation Factor (to the telescope), we neglict the scattering contribution to attenuation for simplicity
    attenuation = np.exp(-(tau_below_atm[:, :, None]) * m_telescope)

    # 7. FULLY VECTORIZED LAYER INTEGRATION
    # Get physical, stitched 4D sky profile: Shape (Nz, Nf, N_theta, N_phi)
    T_ground = Temperature[0]
    T_in = compute_T_in(theta_grid, z_layers, altitudes, Temperature, T_ground, tau_above_atm, tau_below_atm, 
                        consider_earth_curvature, consider_atmospheric_emission)
    
    # 2D Angular Integral: Sum over theta (axis 2) and phi (axis 3)
    integral_M = np.sum(M * T_in * np.sin(tg), axis=(2, 3)) * dtheta * dphi
    
    # Accumulate layer signals
    C_z = ice_dens_arr * dz[:, None]
    layer_signal = integral_M[:, :, None] * C_z[:, None, :] * attenuation
    
    # Base Multipliers: Shape (Nf, Na)
    C_base = m_telescope * k_4[:, None] * V_squared[None, :]
    
    # Final Sky Temperature: Sum over all z_layers (axis 0)
    T_sky_detected = C_base * np.sum(layer_signal, axis=0)
        
    return T_sky_detected


# ====================================================================
# USER WRAPPERS
# ====================================================================

def compute_T_RJ_ice_CLASS1(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, 
                            radius_eq, aspect_ratio=1.0, stokes_param='I',
                            consider_earth_curvature=True, consider_atmospheric_emission=True):
    """ Computes the brightness temperature using METHOD 1 (Direct Earth Frame Projection). """
    return _compute_T_RJ_ice_CLASS_core(
        frequency, altitudes, Temperature, Pressure, P_water, elevation, 
        ice_density, radius_eq, aspect_ratio, stokes_param,
        consider_earth_curvature, consider_atmospheric_emission, method=1,
       
    )

def compute_T_RJ_ice_CLASS2(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, 
                            radius_eq, aspect_ratio=1.0, stokes_param='I',
                            consider_earth_curvature=True, consider_atmospheric_emission=True):
    """ Computes the brightness temperature using METHOD 2 (Textbook Scattering Plane Rotation). """
    return _compute_T_RJ_ice_CLASS_core(
        frequency, altitudes, Temperature, Pressure, P_water, elevation, 
        ice_density, radius_eq, aspect_ratio, stokes_param,
        consider_earth_curvature, consider_atmospheric_emission, method=2
    )

Tried to vectorized along aspect ratio below but didn't work

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants
from atm_tools import alpha_specific_function

pi = constants.pi 
c = constants.c


#The first part is for SPT 

def create_constant_ice_layer(altitudes, layer_bottom, layer_top, ice_density):
    """
    Create a constant ice layer with specified density between given altitudes.

    Parameters:
    altitudes (numpy array): Array of altitude values in meters.
    layer_bottom (float): Bottom altitude of the ice layer in meters.
    layer_top (float): Top altitude of the ice layer in meters.
    ice_density (float): Density of ice particles in particles/m^3.

    Returns:
    numpy array: Array of particle densities corresponding to the input altitudes.
    """
    n = np.zeros_like(altitudes)  # Initialize an array of zeros
    in_layer = (altitudes >= layer_bottom) & (altitudes <= layer_top)  # Find indices within the layer
    n[in_layer] = ice_density  # Set density for those indices
    return n

def sigma_scattering(frequency, volume, A):
    """
    Calculate the scattering cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The scattering cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_sca = w**4*volume**2*np.abs(A)**2/(6*pi*c**4)
    
    return sigma_sca

def sigma_absorption(frequency, volume, A):
    """
    Calculate the absorption cross section for ice crystals.
    
    #Parameters:
    frequency (float): The frequency of the incident light (in Hz).
    volume (float): The volume of the ice crystal particle (in m^3).
    A (float): A constant that depends on the properties of the ice crystal (dielectric properties, shape, polarizability).
    
    Returns:
    float: The absorption cross section (in m^2).
    """
    w = 2 * pi * frequency #angular frequency

    sigma_abs = w*volume*np.imag(A)/c

    return sigma_abs

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) along the symmetry axis
    for a spheroid given its aspect ratio.
    
    Parameters:
    aspect_ratio (float): Ratio of symmetry axis to transverse axis (m).
                          m = 1.0 (sphere), m > 1.0 (column), m < 1.0 (plate).
    
    Returns:
    float: Depolarization factor Delta.
    """
    m = np.atleast_1d(aspect_ratio).astype(float)
    delta = np.zeros_like(m, dtype=float)

    # Sphere
    mask_sphere = (m == 1.0)
    if np.any(mask_sphere):
        delta[mask_sphere] = 1.0 / 3.0

    # Prolate spheroid (Column)
    mask_prolate = (m > 1.0)
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)

    # Oblate spheroid (Plate)
    mask_oblate = (m < 1.0)
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))

    return delta[0] if delta.size == 1 else delta

def compute_intrinsic_polarizabilities(frequency, aspect_ratio):
    """Computes the inherent polarizabilities (A_par, A_perp) relative to the particle's symmetry axis."""
    freq = np.atleast_1d(frequency)
    ar = np.atleast_1d(aspect_ratio)

    eps_prime = 3.16
    eps_double_prime = 8e-3 * (freq / 150e9)
    eps = eps_prime + 1j * eps_double_prime

    # Broadcast to (Nf, Na)
    eps = eps[:, None]             # (Nf, 1)
    ar_b = ar[None, :]             # (1, Na)

    delta = compute_depolarization_factor(ar_b)  # returns (Na,) or (1,Na)
    delta = np.atleast_1d(delta)
    if delta.ndim == 1:
        delta = delta[None, :]

    # Prepare output arrays
    A_par = np.zeros_like(eps, dtype=complex)
    A_perp = np.zeros_like(eps, dtype=complex)

    mask_sphere = (ar_b == 1.0)
    if np.any(mask_sphere):
        A_sphere = 3 * (eps - 1) / (eps + 2)
        A_par[:, mask_sphere.ravel()] = A_sphere[:, 0][:, None]
        A_perp[:, mask_sphere.ravel()] = A_sphere[:, 0][:, None]

    mask_prolate = (ar_b > 1.0)
    if np.any(mask_prolate):
        delta_p = delta[:, mask_prolate.ravel()]
        A_par[:, mask_prolate.ravel()] = (eps - 1) / (1 + (eps - 1) * delta_p)
        A_perp[:, mask_prolate.ravel()] = (eps - 1) / (1 + (eps - 1) * (1 - delta_p) / 2.0)

    mask_oblate = (ar_b < 1.0)
    if np.any(mask_oblate):
        delta_o = delta[:, mask_oblate.ravel()]
        A_par[:, mask_oblate.ravel()] = (eps - 1) / (1 + (eps - 1) * delta_o)
        A_perp[:, mask_oblate.ravel()] = (eps - 1) / (1 + (eps - 1) * (1 - delta_o) / 2.0)

    # Squeeze outputs when inputs were scalars
    if np.ndim(frequency) == 0 and np.ndim(aspect_ratio) == 0:
        return A_par[0, 0], A_perp[0, 0]
    return A_par, A_perp


def compute_effective_polarizability(frequency, aspect_ratio, elevation, stokes_param='I'):
    """
    Projects the polarizabilities onto the telescope based on aerodynamics and elevation.
    Correctly handles the incoherent azimuthal averaging for spinning columns.
    """
    # 1. Get intrinsic complex polarizabilities (supports array outputs)
    A_par, A_perp = compute_intrinsic_polarizabilities(frequency, aspect_ratio)

    # 2. Pre-calculate the physical intensities (abs^2) and absorption (imag)
    abs2_par = np.abs(A_par)**2
    abs2_perp = np.abs(A_perp)**2

    imag_par = np.imag(A_par)
    imag_perp = np.imag(A_perp)

    # Ensure aspect_ratio broadcast shape
    ar = np.atleast_1d(aspect_ratio)
    ar_b = ar[None, :] if ar.ndim == 1 else np.atleast_1d(ar)

    # 3. Map to Earth-frame based on aerodynamics (Incoherent Averaging)
    # All arrays have shape (Nf, Na) after compute_intrinsic_polarizabilities when inputs were arrays
    abs2_v = np.empty_like(abs2_par)
    abs2_h = np.empty_like(abs2_par)
    imag_v = np.empty_like(imag_par)
    imag_h = np.empty_like(imag_par)

    # Sphere
    mask_sphere = (ar == 1.0)
    if np.any(mask_sphere):
        abs2_v[:, mask_sphere] = abs2_par[:, mask_sphere]
        abs2_h[:, mask_sphere] = abs2_par[:, mask_sphere]
        imag_v[:, mask_sphere] = imag_par[:, mask_sphere]
        imag_h[:, mask_sphere] = imag_par[:, mask_sphere]

    # Plates
    mask_plate = (ar < 1.0)
    if np.any(mask_plate):
        abs2_v[:, mask_plate] = abs2_par[:, mask_plate]
        abs2_h[:, mask_plate] = abs2_perp[:, mask_plate]
        imag_v[:, mask_plate] = imag_par[:, mask_plate]
        imag_h[:, mask_plate] = imag_perp[:, mask_plate]

    # Columns
    mask_column = (ar > 1.0)
    if np.any(mask_column):
        abs2_v[:, mask_column] = abs2_perp[:, mask_column]
        abs2_h[:, mask_column] = 0.5 * (abs2_par[:, mask_column] + abs2_perp[:, mask_column])
        imag_v[:, mask_column] = imag_perp[:, mask_column]
        imag_h[:, mask_column] = 0.5 * (imag_par[:, mask_column] + imag_perp[:, mask_column])

    # 4. Project onto Telescope Sensors based on Stokes parameter
    eps_rad = np.radians(elevation)

    if stokes_param == 'I':
        eff_abs2 = 0.5 * (abs2_v * np.cos(eps_rad)**2 + abs2_h * (1.0 + np.sin(eps_rad)**2))
        eff_imag = 0.5 * (imag_v * np.cos(eps_rad)**2 + imag_h * (1.0 + np.sin(eps_rad)**2))

    elif stokes_param == 'Q':
        eff_abs2 = 0.5 * (abs2_v - abs2_h) * np.cos(eps_rad)**2
        eff_imag = 0.5 * (imag_v - imag_h) * np.cos(eps_rad)**2

    else:
        raise ValueError("stokes_param must be 'I' or 'Q'")

    # Squeeze outputs when inputs were scalars
    if np.ndim(frequency) == 0 and np.ndim(aspect_ratio) == 0:
        return np.squeeze(eff_abs2), np.squeeze(eff_imag)

    return eff_abs2, eff_imag

def polarization_fraction_2(frequency, elevation_deg, aspect_ratio=0.5, process='scattering'):
    """
    Computes the polarization fraction (Q/I) directly from the effective polarizabilities,
    bypassing the intermediate gamma ratio calculation.
    
    Parameters:
    frequency (float or array): Frequency in Hz.
    elevation_deg (float or array): Telescope elevation angle in degrees.
    aspect_ratio (float): Ratio of symmetry axis to transverse axis. Defaults to 0.5 (plate).
    process (str): 'scattering' or 'emission'.
    
    Returns:
    float or array: The polarization fraction p_gamma.
    """
    
    # 1. Get the effective polarizabilities for Total Intensity (I)
    eff_abs2_I, eff_imag_I = compute_effective_polarizability(
        frequency, aspect_ratio, elevation_deg, stokes_param='I'
    )

    # 2. Get the effective polarizabilities for Linear Polarization (Q)
    eff_abs2_Q, eff_imag_Q = compute_effective_polarizability(
        frequency, aspect_ratio, elevation_deg, stokes_param='Q'
    )
    
    # 3. Calculate the fraction depending on the physical process
    if process == 'scattering':
        # Scattering scales with the absolute square of the polarizability
        p_frac = eff_abs2_Q / eff_abs2_I
        
    elif process == 'emission':
        # Thermal emission scales with the imaginary part of the polarizability (absorption)
        p_frac = eff_imag_Q / eff_imag_I
        
    else:
        raise ValueError("Process must be 'scattering' or 'emission'.")
        
    return p_frac



def compute_T_RJ_ice2(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, radius_eq, aspect_ratio=1.0, process='total', stokes_param='I'):
    """
    Computes the Rayleigh-Jeans brightness temperature contribution from ice crystals 
    along a line of sight for a specific Stokes parameter (I or Q).
    
    Parameters:
    frequency (array-like): Array of frequencies in Hz. (Nf)
    altitudes (array-like): Array of altitudes in meters. (Nz)
    Temperature (array-like): Array of atmospheric temperatures in K. (Nz)
    Pressure (array-like): Array of atmospheric pressures in hPa. (Nz)
    P_water (array-like): Array of water vapor partial pressures in hPa. (Nz)
    elevation (float): Elevation angle of the telescope in degrees.
    ice_density (array-like): Array of ice water content (in particles/m^3). (Nz, Na)
    radius_eq (array-like): Array of equivalent radii of the ice crystals in meters. (Na)
    aspect_ratio (float or array): Ratio of symmetry axis to transverse axis.
    process (str): 'scattering', 'emission', or 'total'.
    stokes_param (str): 'I' (Total Intensity) or 'Q' (Linear Polarization).

    Returns:
    ndarray: The Rayleigh-Jeans brightness temperature array (shape: Nf, Na).
    """
    c = constants.c
    
    # 1. Geometry and airmass
    zenith_angle = 90.0 - elevation
    m = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    dz = np.diff(altitudes) # Shape: (Nz-1,)

    # Particle volumes
    V = (4.0 / 3.0) * np.pi * radius_eq**3 # Shape: (Na,)
    
    # 2. Get Effective Cross-Section Multipliers
    eff_abs2_sca_req, eff_imag_abs_req = compute_effective_polarizability(
        frequency, aspect_ratio, elevation, stokes_param=stokes_param
    )
    eff_abs2_sca_I, eff_imag_abs_I = compute_effective_polarizability(
        frequency, aspect_ratio, elevation, stokes_param='I'
    )

    # 3. Calculate Ice Cross-Sections and Alphas
    k_4 = (2 * np.pi * frequency[:, None] / c)**4
    k_1 = (2 * np.pi * frequency[:, None] / c)
    geom_factor = 1.0 / (6.0 * np.pi)

    # Ensure shapes: k_4 (Nf,), V (Na,), eff arrays (Nf,Na)
    sigma_sca_req = geom_factor * (k_4[:, None]) * (V[None, :]**2) * eff_abs2_sca_req
    sigma_sca_I   = geom_factor * (k_4[:, None]) * (V[None, :]**2) * eff_abs2_sca_I

    sigma_abs_req = (k_1[:, None]) * V[None, :] * eff_imag_abs_req
    sigma_abs_I   = (k_1[:, None]) * V[None, :] * eff_imag_abs_I

    alpha_sca_req = ice_density[:, None, :] * sigma_sca_req[None, :, :] 
    alpha_sca_I   = ice_density[:, None, :] * sigma_sca_I[None, :, :] 
    alpha_abs_req = ice_density[:, None, :] * sigma_abs_req[None, :, :] 
    alpha_abs_I   = ice_density[:, None, :] * sigma_abs_I[None, :, :] 

    # Midpoint averages for ice layers
    alpha_sca_req_mid = (alpha_sca_req[:-1, :, :] + alpha_sca_req[1:, :, :]) / 2.0
    alpha_sca_I_mid   = (alpha_sca_I[:-1, :, :] + alpha_sca_I[1:, :, :]) / 2.0
    alpha_abs_req_mid = (alpha_abs_req[:-1, :, :] + alpha_abs_req[1:, :, :]) / 2.0
    alpha_abs_I_mid   = (alpha_abs_I[:-1, :, :] + alpha_abs_I[1:, :, :]) / 2.0

    # Optical thicknesses for ice: Shape (Nz-1, Nf, Na)
    d_tau_sca_req = alpha_sca_req_mid * dz[:, None, None]
    d_tau_sca_I   = alpha_sca_I_mid * dz[:, None, None]
    d_tau_abs_req = alpha_abs_req_mid * dz[:, None, None]
    d_tau_abs_I   = alpha_abs_I_mid * dz[:, None, None]

    # Total Ice Optical Depth (Intensity) for cumulative attenuation
    d_tau_ext_I = d_tau_sca_I + d_tau_abs_I
    
    # Cumulative ice attenuation from below
    tau_below_ice = np.cumsum(d_tau_ext_I, axis=0)
    tau_below_ice = np.insert(tau_below_ice[:-1, :, :], 0, 0, axis=0) 

    # 4. Atmospheric (Gas) Optical Depths
    alpha_atm = alpha_specific_function(altitudes, frequency, Temperature, Pressure, P_water) # Shape: (Nz, Nf)
    alpha_atm_mid = (alpha_atm[:-1, :] + alpha_atm[1:, :]) / 2.0
    
    d_tau_atm = alpha_atm_mid * dz[:, None] # Shape: (Nz-1, Nf)
    tau_below_atm = np.cumsum(d_tau_atm, axis=0)
    tau_below_atm = np.insert(tau_below_atm[:-1, :], 0, 0, axis=0) 

    # Midpoint Thermodynamic Temperatures
    T_mid = (Temperature[:-1] + Temperature[1:]) / 2.0 # Shape: (Nz-1,)

    # 5. The Universal Attenuation Factor to reach the telescope
    # Broadcast tau_below_atm to match ice dimensions (Nz-1, Nf, Na)
    attenuation = np.exp(-(tau_below_ice + tau_below_atm[:, :, None]) * m)

    # 6. Radiative Transfer Integration
    T_layers_sca = 0.0
    T_layers_abs = 0.0

    if process in ['scattering', 'total']:
        # Source is ground temperature (approximated half-hemisphere)
        T_source_sca = Temperature[0] / 2.0 
        
        p_gamma_sca = np.divide(d_tau_sca_req, d_tau_sca_I, out=np.zeros_like(d_tau_sca_I), where=(d_tau_sca_I != 0))
        
        # Radiative Transfer: Source * Generation * Attenuation
        T_layers_sca_I = T_source_sca * (1 - np.exp(-d_tau_sca_I * m))
        T_layers_sca = p_gamma_sca * T_layers_sca_I * attenuation

    if process in ['emission', 'total']:
        T_source_abs = T_mid[:, None, None]
        
        p_gamma_abs = np.divide(d_tau_abs_req, d_tau_abs_I, out=np.zeros_like(d_tau_abs_I), where=(d_tau_abs_I != 0))
        
        # Radiative Transfer: Source * Generation * Attenuation
        T_layers_abs_I = T_source_abs * (1 - np.exp(-d_tau_abs_I * m))
        T_layers_abs = p_gamma_abs * T_layers_abs_I * attenuation

    # 7. Selection and Summation
    if process == 'scattering':
        T_layers = T_layers_sca
    elif process == 'emission':
        T_layers = T_layers_abs
    else: 
        T_layers = T_layers_sca + T_layers_abs

    T_sky = np.sum(T_layers, axis=0) 
    
    return T_sky





#Let's go over the CLASS code

#The brightness temperature seen by an ice crystal

import numpy as np

def compute_T_in_1D(theta_grid, z_part, altitudes, T_phys_profile, T_ground, tau_z_profile):
    """
    Computes the full angular profile of incoming brightness temperature hitting a particle.
    
    Parameters:
    theta_grid     : ND array of zenith angles in radians (e.g., shape (1, N_theta, N_phi))
    z_part         : Scalar, altitude of the active layer/particle in meters
    altitudes      : 1D array of atmospheric profile altitudes in meters (ascending)
    T_phys_profile : 1D array of thermodynamic temperatures corresponding to altitudes
    T_ground       : Scalar, physical temperature of the ground
    tau_z_profile  : 1D array, cumulative optical depth from the ground up to 'altitudes'
    """
    R_e = 6371e3  # Earth radius in meters
    
    # ---------------------------------------------------------
    # 1. EXTRACT LOCAL LAYER PROPERTIES
    # ---------------------------------------------------------
    # Interpolate to find exactly what is happening at z_part
    tau_below = np.interp(z_part, altitudes, tau_z_profile)
    tau_total = tau_z_profile[-1]
    tau_above = tau_total - tau_below
    
    # Estimate effective temperatures of the atmosphere above and below the particle
    mask_above = altitudes >= z_part
    mask_below = altitudes <= z_part
    T_atm_above = np.mean(T_phys_profile[mask_above]) if np.any(mask_above) else T_phys_profile[-1]
    T_atm_below = np.mean(T_phys_profile[mask_below]) if np.any(mask_below) else T_ground

    # ---------------------------------------------------------
    # 2. DEFINE THE ANGULAR BOUNDARIES
    # ---------------------------------------------------------
    theta_h = np.sqrt(2 * z_part / R_e)
    bound_sky = np.pi / 2.0
    bound_limb = (np.pi / 2.0) + theta_h

    # ---------------------------------------------------------
    # 3. ZONE 1: THE UPPER SKY (theta < pi/2)
    # ---------------------------------------------------------
    # Kasten-Young empirical airmass
    theta_deg = np.degrees(theta_grid)
    theta_deg_sky = np.clip(theta_deg, 0, 90.0) 
    m_sky = 1.0 / (np.cos(theta_grid) + 0.50572 * (96.07995 - theta_deg_sky)**(-1.6364))
    
    T_sky = T_atm_above * (1.0 - np.exp(-tau_above * m_sky))

    # ---------------------------------------------------------
    # 4. ZONE 2: THE TANGENT LIMB (pi/2 <= theta <= pi/2 + theta_h)
    # ---------------------------------------------------------
    # Calculate the minimum grazing altitude of the ray
    z_min_grid = (R_e + z_part) * np.sin(theta_grid) - R_e
    
    # Interpolate physical temperature at the grazing altitude
    T_limb_flat = np.interp(z_min_grid.flatten(), altitudes, T_phys_profile)
    T_limb = T_limb_flat.reshape(theta_grid.shape)

    # ---------------------------------------------------------
    # 5. ZONE 3: THE GROUND (theta > pi/2 + theta_h)
    # ---------------------------------------------------------
    # For looking down, we use the downward secant law: 1 / cos(pi - theta)
    # We clip the angle slightly away from exactly 90 deg to prevent division by zero
    theta_down = np.clip(theta_grid, bound_limb + 1e-4, np.pi)
    m_down = 1.0 / np.cos(np.pi - theta_down)
    
    # Emission from the ground (attenuated) + emission from the air below the particle
    T_ground_zone = T_ground * np.exp(-tau_below * m_down) + \
                    T_atm_below * (1.0 - np.exp(-tau_below * m_down))

    # ---------------------------------------------------------
    # 6. STITCH THE ZONES TOGETHER
    # ---------------------------------------------------------
    conditions = [
        theta_grid < bound_sky,
        (theta_grid >= bound_sky) & (theta_grid <= bound_limb),
        theta_grid > bound_limb
    ]
    
    choices = [T_sky, T_limb, T_ground_zone]
    
    T_in = np.select(conditions, choices)
    
    return T_in

def compute_depolarization_factor(aspect_ratio):
    """
    Calculates the depolarization factor (Delta) for the SYMMETRY axis 
    of a spheroid based on its aspect ratio.
    """
    m = np.atleast_1d(aspect_ratio)
    delta = np.zeros_like(m, dtype=float)
    
    mask_sphere = (m == 1.0)
    delta[mask_sphere] = 1.0 / 3.0
    
    mask_prolate = (m > 1.0)  # Columns
    if np.any(mask_prolate):
        m_p = m[mask_prolate]
        e = np.sqrt(1.0 - (1.0 / m_p)**2)
        delta[mask_prolate] = ((1.0 - e**2) / e**2) * ((1.0 / (2.0 * e)) * np.log((1.0 + e) / (1.0 - e)) - 1.0)
        
    mask_oblate = (m < 1.0)   # Plates
    if np.any(mask_oblate):
        m_o = m[mask_oblate]
        e = np.sqrt(1.0 - m_o**2)
        delta[mask_oblate] = (1.0 / e**2) * (1.0 - (np.sqrt(1.0 - e**2) / e) * np.arcsin(e))
        
    return delta[0] if delta.size == 1 else delta


import numpy as np

def compute_polarizability(frequency, aspect_ratio=1.0):
    """
    Computes the complex polarizabilities (alpha_h, alpha_v) for ice crystals.
    Constructs an effective complex alpha_h for columns to mathematically respect 
    the incoherent azimuthal averaging of the absolute squares.
    """
    freq = np.atleast_1d(frequency)
    ar = np.atleast_1d(aspect_ratio)

    eps_prime = 3.16
    eps_double_prime = 8e-3 * (freq / 150e9)
    eps = eps_prime + 1j * eps_double_prime

    # Broadcast to (Nf, Na)
    eps = eps[:, None]         # (Nf,1)
    ar_b = ar[None, :]         # (1,Na)

    delta = compute_depolarization_factor(ar_b)
    delta = np.atleast_1d(delta)
    if delta.ndim == 1:
        delta = delta[None, :]

    A_par = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * delta))
    A_perp = (eps - 1) / (4 * np.pi * (1 + (eps - 1) * (1 - delta) / 2.0))

    # Prepare outputs
    alpha_h = np.zeros_like(A_par, dtype=complex)
    alpha_v = np.zeros_like(A_par, dtype=complex)

    mask_sphere = (ar_b == 1.0)
    if np.any(mask_sphere):
        alpha_v[:, mask_sphere.ravel()] = A_par[:, mask_sphere.ravel()]
        alpha_h[:, mask_sphere.ravel()] = A_par[:, mask_sphere.ravel()]

    mask_plate = (ar_b < 1.0)
    if np.any(mask_plate):
        alpha_v[:, mask_plate.ravel()] = A_par[:, mask_plate.ravel()]
        alpha_h[:, mask_plate.ravel()] = A_perp[:, mask_plate.ravel()]

    mask_column = (ar_b > 1.0)
    if np.any(mask_column):
        alpha_v[:, mask_column.ravel()] = A_perp[:, mask_column.ravel()]

        target_abs2_h = 0.5 * (np.abs(A_par[:, mask_column.ravel()])**2 + np.abs(A_perp[:, mask_column.ravel()])**2)
        target_imag_h = 0.5 * (np.imag(A_par[:, mask_column.ravel()]) + np.imag(A_perp[:, mask_column.ravel()]))

        real_h = np.sqrt(np.maximum(0.0, target_abs2_h - target_imag_h**2))
        alpha_h[:, mask_column.ravel()] = real_h + 1j * target_imag_h

    # Squeeze for scalar inputs
    if np.ndim(frequency) == 0 and np.ndim(aspect_ratio) == 0:
        return alpha_h[0, 0], alpha_v[0, 0]

    return alpha_h, alpha_v

import numpy as np

# ====================================================================
# METHOD 1: DIRECT EARTH FRAME (Fast & Elegant)
# ====================================================================
def compute_earth_frame_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    abs_h2 = np.abs(alpha_h)**2
    abs_v2 = np.abs(alpha_v)**2
    
    # Scattered intensity projected onto the Horizontal telescope axis
    I_H = abs_h2 * (1.0 - (np.sin(theta_grid) * np.sin(phi_grid))**2)
    
    # Scattered intensity projected onto the Vertical telescope axis
    ray_proj = alpha_v * np.cos(delta) * np.cos(theta_grid) - \
               alpha_h * np.sin(delta) * np.sin(theta_grid) * np.cos(phi_grid)
    I_V = (abs_h2 * np.sin(delta)**2 + abs_v2 * np.cos(delta)**2) - np.abs(ray_proj)**2
    
    # Calculate M11 (Total Intensity) and M21 (Linear Polarization Q)
    M11_earth = 0.5 * (I_V + I_H)
    M21_earth = 0.5 * (I_V - I_H)
    
    return M11_earth, M21_earth


# ====================================================================
# METHOD 2: TEXTBOOK SCATTERING PLANE (Corrected Vector Math)
# ====================================================================
def compute_scattering_plane_phase_matrix(alpha_h, alpha_v, theta_grid, phi_grid, delta):
    
    # 1. Define Propagation Vectors (k_i and k_s)
    s_i = np.stack([-np.sin(theta_grid) * np.cos(phi_grid), 
                    -np.sin(theta_grid) * np.sin(phi_grid), 
                    -np.cos(theta_grid)], axis=-1)
    
    # Propagating down to the telescope
    s_s = np.array([-np.cos(delta), 0.0, -np.sin(delta)])
    s_s = np.broadcast_to(s_s, s_i.shape)
    
    # 2. Build the 2D Scattering Plane Unit Vectors
    n_scat = np.cross(s_i, s_s, axis=-1)
    norm = np.linalg.norm(n_scat, axis=-1, keepdims=True)
    norm = np.where(norm < 1e-12, 1e-12, norm) 
    e_perp = n_scat / norm
    
    # Strict Right-Hand Rule (e_par = e_perp x k)
    e_par_i = np.cross(e_perp, s_i, axis=-1)
    e_par_s = np.cross(e_perp, s_s, axis=-1)
    
    # 3. Apply the Earth-locked Ice Crystal Tensor
    def apply_tensor(v):
        return np.stack([alpha_h * v[..., 0], alpha_h * v[..., 1], alpha_v * v[..., 2]], axis=-1)
    
    # 4. Calculate Bohren & Huffman Amplitude Matrix Elements
    S1 = np.sum(e_perp * apply_tensor(e_perp), axis=-1)
    S2 = np.sum(e_par_s * apply_tensor(e_par_i), axis=-1)
    S3 = np.sum(e_par_s * apply_tensor(e_perp), axis=-1)
    S4 = np.sum(e_perp * apply_tensor(e_par_i), axis=-1)
    
    # 5. Stokes Phase Matrix inside the local scattering plane (F)
    # Added F11 which sums all power states
    F11 = 0.5 * (np.abs(S1)**2 + np.abs(S2)**2 + np.abs(S3)**2 + np.abs(S4)**2)
    F21 = 0.5 * (np.abs(S2)**2 + np.abs(S3)**2 - np.abs(S1)**2 - np.abs(S4)**2)
    F31 = np.real(S2 * np.conj(S4) + S3 * np.conj(S1))
    
    # 6. EXACT Rotation: Bypass spherical trig and use exact 3D Dot Products
    e_v = np.array([-np.sin(delta), 0.0, np.cos(delta)])
    e_v = np.broadcast_to(e_v, s_i.shape)
    
    # Project the telescope vertical sensor onto the scattering plane basis
    cos_psi = np.sum(e_v * e_par_s, axis=-1)
    sin_psi = -np.sum(e_v * e_perp, axis=-1)
    
    # Apply double-angle formulas for the Mueller rotation matrix R(psi)
    cos_2psi = cos_psi**2 - sin_psi**2
    sin_2psi = 2 * sin_psi * cos_psi
    
    # M11 bypasses rotation entirely. M21 is rotated into the telescope frame.
    M11_rotated = F11 
    M21_rotated = F21 * cos_2psi - F31 * sin_2psi
    
    return M11_rotated, M21_rotated


import numpy as np
from scipy import constants

# ====================================================================
# HELPER: VECTORIZED 3D SKY TEMPERATURE
# ====================================================================
"""
def compute_T_in(theta_grid, z_layers, altitudes, T_phys_profile, T_ground, tau_above_atm, tau_below_atm):
    
    #Computes the incoming brightness temperature hitting the ice crystal from all angles.
    #Fully vectorized across layers (Nz), frequencies (Nf), and angles (N_theta, N_phi).
    
    R_e = 6371e3
    
    # Expand physical arrays for 4D broadcasting: (Nz, Nf, N_theta, N_phi)
    # tg shape: (1, 1, N_theta, N_phi)
    tg = theta_grid[None, None, :, :] 
    z_exp = z_layers[:, None, None, None]
    T_mid = ((T_phys_profile[:-1] + T_phys_profile[1:]) / 2.0)[:, None, None, None]
    
    tau_above = tau_above_atm[:, :, None, None]
    tau_below = tau_below_atm[:, :, None, None]
    
    # Geometric Boundaries
    theta_h = np.sqrt(2 * z_exp / R_e)
    bound_sky = np.pi / 2
    bound_limb = (np.pi / 2) + theta_h
    
    # 1. SKY REGIME (Theta < 90 deg)
    # Using Kasten-Young airmass, clipped exactly at 90 deg
    theta_deg = np.clip(np.degrees(tg), 0, 90.0)
    m_sky = 1.0 / (np.cos(tg) + 0.50572 * (96.07995 - theta_deg)**(-1.6364))
    T_sky = T_mid * (1.0 - np.exp(-tau_above * m_sky))
    
    # 2. LIMB REGIME (90 <= Theta <= 90 + theta_h)
    # Saturated tangent ray takes the physical temperature of its minimum altitude
    z_min = (R_e + z_exp) * np.sin(tg) - R_e
    T_limb_flat = np.interp(z_min.flatten(), altitudes, T_phys_profile)
    T_limb = T_limb_flat.reshape(z_min.shape)
    
    # 3. GROUND REGIME (Theta > 90 + theta_h)
    # Upwelling attenuated ground emission + lower atmosphere emission
    m_gnd = 1.0 / np.maximum(np.cos(np.pi - tg), 0.01) # Avoid div by zero looking horizontally
    T_gnd_attenuated = T_ground * np.exp(-tau_below * m_gnd) + T_mid * (1.0 - np.exp(-tau_below * m_gnd))
    
    # Stitch the regimes together
    conditions = [
        tg < bound_sky,
        (tg >= bound_sky) & (tg <= bound_limb),
        tg > bound_limb
    ]
    choices = [T_sky, T_limb, T_gnd_attenuated]
    
    return np.select(conditions, choices)
"""

def compute_T_in(theta_grid, z_layers, altitudes, T_phys_profile, T_ground, tau_above_atm, tau_below_atm, 
                 consider_earth_curvature=True, consider_atmospheric_emission=True):
    """
    Computes the incoming brightness temperature hitting the ice crystal from all angles.
    Fully vectorized across layers (Nz), frequencies (Nf), and angles (N_theta, N_phi).
    
    Toggles:
    - consider_earth_curvature: If False, assumes a flat Earth (no limb regime).
    - consider_atmospheric_emission: If False, ignores atmospheric emission/attenuation 
                                     (Sky = 0K, Ground = pure T_ground).
    """
    R_e = 6371e3
    
    # Expand physical arrays for 4D broadcasting: (Nz, Nf, N_theta, N_phi)
    tg = theta_grid[None, None, :, :] 
    z_exp = z_layers[:, None, None, None]
    T_mid = ((T_phys_profile[:-1] + T_phys_profile[1:]) / 2.0)[:, None, None, None]
    
    tau_above = tau_above_atm[:, :, None, None]
    tau_below = tau_below_atm[:, :, None, None]
    
    # ==========================================
    # 1. BASE PHYSICS (Atmosphere vs No Atmosphere)
    # ==========================================
    if consider_atmospheric_emission:
        # Sky emission (downwelling)
        theta_deg = np.clip(np.degrees(tg), 0, 90.0)
        m_sky = 1.0 / (np.cos(tg) + 0.50572 * (96.07995 - theta_deg)**(-1.6364))
        T_sky = T_mid * (1.0 - np.exp(-tau_above * m_sky))
        
        # Ground emission (upwelling + atmospheric attenuation/emission)
        m_gnd = 1.0 / np.maximum(np.cos(np.pi - tg), 0.01)
        T_gnd_effective = T_ground * np.exp(-tau_below * m_gnd) + T_mid * (1.0 - np.exp(-tau_below * m_gnd))
    else:
        # No atmosphere: Sky is dark space (0K), Ground is unattenuated
        T_sky = np.zeros_like(tg)
        T_gnd_effective = np.full_like(tg, T_ground)

    # ==========================================
    # 2. GEOMETRIC STITCHING (Curved vs Flat Earth)
    # ==========================================
    if consider_earth_curvature:
        theta_h = np.sqrt(2 * z_exp / R_e)
        bound_sky = np.pi / 2
        bound_limb = (np.pi / 2) + theta_h
        
        if consider_atmospheric_emission:
            # Limb rays saturate to the physical temperature of their lowest atmospheric dip
            z_min = (R_e + z_exp) * np.sin(tg) - R_e
            T_limb_flat = np.interp(z_min.flatten(), altitudes, T_phys_profile)
            T_limb = T_limb_flat.reshape(z_min.shape)
        else:
            # Without atmosphere, limb rays that graze the Earth just hit deep space
            T_limb = np.zeros_like(tg) 
            
        conditions = [
            tg <= bound_sky,
            (tg > bound_sky) & (tg <= bound_limb),
            tg > bound_limb
        ]
        choices = [T_sky, T_limb, T_gnd_effective]
        
    else:
        # Flat Earth Approximation (Only Two Regimes)
        bound_sky = np.pi / 2
        
        conditions = [
            tg <= bound_sky,
            tg > bound_sky
        ]
        choices = [T_sky, T_gnd_effective]
    
    return np.select(conditions, choices)

# ====================================================================
# CLASS INTEGRATION ENGINE (Fully Coupled with Atmosphere)
# ====================================================================

def _compute_T_RJ_ice_CLASS_core(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, 
                                 radius_eq, aspect_ratio, stokes_param, 
                                 consider_earth_curvature, consider_atmospheric_emission, method):
    c = constants.c
    delta = np.radians(elevation)
    
    # 1. Kasten-Young Airmass for Telescope Line of Sight
    zenith_angle = 90.0 - elevation
    m_telescope = 1.0 / (np.cos(np.radians(zenith_angle)) + 0.50572 * (96.07995 - zenith_angle)**(-1.6364))
    
    # 2. Standardize Inputs & Grid Alignment
    freq_arr = np.atleast_1d(frequency)
    radius_arr = np.atleast_1d(radius_eq)
    
    ice_dens_arr = np.asarray(ice_density)
    if ice_dens_arr.ndim == 1:
        ice_dens_arr = ice_dens_arr[:, None] 
        
    if ice_dens_arr.shape[0] == len(altitudes):
        ice_dens_arr = (ice_dens_arr[:-1, :] + ice_dens_arr[1:, :]) / 2.0
        
    dz = np.diff(altitudes)
    z_layers = (altitudes[:-1] + altitudes[1:]) / 2.0
    
    # 3. Angular Grid Setup
    N_theta, N_phi = 100, 100
    theta = np.linspace(0, np.pi, N_theta)
    phi = np.linspace(0, 2*np.pi, N_phi, endpoint=False)
    dtheta, dphi = theta[1] - theta[0], phi[1] - phi[0]
    
    theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')
    tg = theta_grid[None, :, :]
    pg = phi_grid[None, :, :]
    
    # 4. Ice Physics Multipliers
    alpha_h, alpha_v = compute_polarizability(freq_arr, aspect_ratio) 
    alpha_h_exp = np.atleast_1d(alpha_h)[:, None, None]
    alpha_v_exp = np.atleast_1d(alpha_v)[:, None, None]
    
    V = (4.0 / 3.0) * np.pi * radius_arr**3
    V_squared = (16.0 / 9.0) * np.pi**2 * (radius_arr)**6
    k_4 = (2 * np.pi * freq_arr / c)**4
    
    if method == 1:
        M11, M21 = compute_earth_frame_phase_matrix(alpha_h_exp, alpha_v_exp, tg, pg, delta)
    else:
        M11, M21 = compute_scattering_plane_phase_matrix(alpha_h_exp, alpha_v_exp, tg, pg, delta)
        
    M = M11 if stokes_param == 'I' else M21
    
    # 5. Atmospheric (Gas) Optical Depths
    alpha_atm = alpha_specific_function(altitudes, freq_arr, Temperature, Pressure, P_water)
    alpha_atm_mid = (alpha_atm[:-1, :] + alpha_atm[1:, :]) / 2.0
    d_tau_atm = alpha_atm_mid * dz[:, None] 
    
    # Attenuation from below (for telescope sightline & ground emission)
    tau_below_atm = np.cumsum(d_tau_atm, axis=0)
    tau_below_atm = np.insert(tau_below_atm[:-1, :], 0, 0, axis=0) 
    
    # Attenuation from above (for sky emission hitting the particle)
    tau_above_atm = np.zeros_like(d_tau_atm)
    if len(d_tau_atm) > 1:
        tau_above_atm[:-1, :] = np.cumsum(d_tau_atm[1:][::-1], axis=0)[::-1]
    
    # The Universal Attenuation Factor (to the telescope), we neglict the scattering contribution to attenuation for simplicity
    attenuation = np.exp(-(tau_below_atm[:, :, None]) * m_telescope)

    # 7. FULLY VECTORIZED LAYER INTEGRATION
    # Get physical, stitched 4D sky profile: Shape (Nz, Nf, N_theta, N_phi)
    T_ground = Temperature[0]
    T_in = compute_T_in(theta_grid, z_layers, altitudes, Temperature, T_ground, tau_above_atm, tau_below_atm, 
                        consider_earth_curvature, consider_atmospheric_emission)
    
    # 2D Angular Integral: Sum over theta (axis 2) and phi (axis 3)
    integral_M = np.sum(M * T_in * np.sin(tg), axis=(2, 3)) * dtheta * dphi
    
    # Accumulate layer signals
    C_z = ice_dens_arr * dz[:, None]
    layer_signal = integral_M[:, :, None] * C_z[:, None, :] * attenuation
    
    # Base Multipliers: Shape (Nf, Na)
    C_base = m_telescope * k_4[:, None] * V_squared[None, :]
    
    # Final Sky Temperature: Sum over all z_layers (axis 0)
    T_sky_detected = C_base * np.sum(layer_signal, axis=0)
        
    return T_sky_detected


# ====================================================================
# USER WRAPPERS
# ====================================================================

def compute_T_RJ_ice_CLASS1(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, 
                            radius_eq, aspect_ratio=1.0, stokes_param='I',
                            consider_earth_curvature=True, consider_atmospheric_emission=True):
    """ Computes the brightness temperature using METHOD 1 (Direct Earth Frame Projection). """
    return _compute_T_RJ_ice_CLASS_core(
        frequency, altitudes, Temperature, Pressure, P_water, elevation, 
        ice_density, radius_eq, aspect_ratio, stokes_param,
        consider_earth_curvature, consider_atmospheric_emission, method=1,
       
    )

def compute_T_RJ_ice_CLASS2(frequency, altitudes, Temperature, Pressure, P_water, elevation, ice_density, 
                            radius_eq, aspect_ratio=1.0, stokes_param='I',
                            consider_earth_curvature=True, consider_atmospheric_emission=True):
    """ Computes the brightness temperature using METHOD 2 (Textbook Scattering Plane Rotation). """
    return _compute_T_RJ_ice_CLASS_core(
        frequency, altitudes, Temperature, Pressure, P_water, elevation, 
        ice_density, radius_eq, aspect_ratio, stokes_param,
        consider_earth_curvature, consider_atmospheric_emission, method=2
    )